# 구조물 안정성 물리 추론 AI 경진대회 - Submission Package


## solution.md


# 구조물 안정성 물리 추론 AI 경진대회 - 솔루션

## 1. 문제 정의 및 접근 전략

### 1.1 문제 요약
- **과제**: 2개 시점(front, top) 이미지로부터 구조물의 붕괴 확률(unstable) vs 안정 확률(stable) 예측
- **핵심 난이도**: 학습 데이터(1,000개)는 고정 환경, 평가 데이터(1,000개)는 무작위 광원/카메라 → **도메인 갭** 극복 필요
- **평가**: LogLoss (확률 calibration이 핵심)

### 1.2 접근 전략
1. **강력한 백본 + 소규모 데이터 최적화**: ConvNeXt-Large (197M params)를 BS=2로 안정 학습
2. **Cross-View Attention**: 두 시점 간 구조적 관계를 명시적으로 모델링
3. **Video Knowledge Distillation**: 시뮬레이션 영상에서 추출한 물리 특징을 이미지 모델에 전이
4. **2-Phase Training**: Supervised → Pseudo Label Fine-tuning으로 도메인 적응
5. **Temperature Scaling**: 제출 피드백 기반 확률 calibration
6. **Multi-Seed Ensemble**: 동일 구조, 다른 시드로 학습한 모델의 기하평균 앙상블

---

## 2. 모델 아키텍처

### 2.1 전체 구조

```
Input: (B, 2, 3, 512, 512)  <- front + top 이미지
            |
    +-------+-------+
    v               v
ConvNeXt-Large  ConvNeXt-Large   (가중치 공유)
    |               |
  1x1 Conv        1x1 Conv       (1536ch -> 512ch 투영)
    |               |
    v               v
  Cross-View Attention (8-head)  <- 두 시점 간 상호 참조
    |               |
  GeM Pool        GeM Pool
    |               |
    +-------+-------+
            |
      [1024] + [8]  <- 영상 물리 피처 concat
            |
      FC -> 512 -> 256 -> 2  (분류 헤드)
            |
        [unstable, stable]
```

### 2.2 핵심 컴포넌트

| 컴포넌트 | 설명 |
|----------|------|
| **Backbone** | `convnext_large.fb_in22k_ft_in1k` (ImageNet-22K pretrained, 197M params) |
| **Cross-View Attention** | Bidirectional Multi-Head Attention (8 heads, dim=512) + FFN. 두 시점의 feature map을 sequence로 펼쳐 상호 참조 |
| **GeM Pooling** | Generalized Mean Pooling (학습 가능 p=3.0). Global Average Pooling 대비 판별력 높은 영역에 가중치 부여 |
| **Video Feature Head** | 시뮬레이션 영상에서 추출한 8차원 물리 피처를 분류 헤드에 직접 concat |
| **Auxiliary Head** | 이미지만으로 motion_score를 회귀 예측하는 보조 손실 (Knowledge Distillation) |

### 2.3 Video Feature (Knowledge Distillation)

학습 데이터에만 존재하는 시뮬레이션 영상(simulation.mp4)에서 Optical Flow 기반 물리 특징 8개를 사전 추출:

| 피처 | 설명 |
|------|------|
| `motion_score` | Farneback Optical Flow 평균 이동량 |
| `frame_diff` | 첫 vs 마지막 프레임 차이 |
| `max_flow` | 최대 픽셀 이동량 |
| `dominant_freq` | FFT 지배 주파수 (진동 주기) |
| `motion_accel` | 운동량 변화율 (가속도) |
| `collapse_frame` | 최대 운동 발생 시점 (정규화) |
| `top_motion` | 상부 구조 움직임 |
| `bottom_motion` | 하부 기반 움직임 |

- 모든 피처는 min-max 정규화 [0, 1]
- 학습 시 50% 확률로 per-sample masking → test 시 피처 없는 상황에 적응
- 보조 헤드가 이미지만으로 motion_score 예측 학습 → 경계 케이스 판별력 향상

---

## 3. 학습 파이프라인

### 3.1 2-Phase Training

#### Phase 1: Supervised 5-Fold CV
- **데이터**: train(1,000) + dev(100) 합산 → StratifiedKFold(5)
- **목적**: 기본 분류 능력 학습
- **설정**: 100 epochs, patience=35, Cosine LR (T_max=100)
- **손실**: CrossEntropyLoss + Soft Label KD (weight=0.7) + Auxiliary MSE (weight=0.3)

#### Phase 2: Pseudo Label Fine-tuning
- **데이터**: Phase1 OOF 예측으로 test 데이터에 pseudo label 부여 (confidence >= 0.80)
- **목적**: test 도메인(무작위 환경) 적응
- **설정**: 60 epochs, patience=20, Cosine LR (T_max=60), LR=5e-5
- **검증**: combined data의 10%를 validation으로 분리

### 3.2 Data Augmentation

도메인 갭(고정 환경 → 무작위 환경) 극복을 위한 강한 augmentation:

| 카테고리 | 기법 |
|----------|------|
| **공간 변환** | Affine(+-10%, +-15deg), Perspective, HorizontalFlip, GridDistortion, ElasticTransform |
| **광원/색상** | RandomBrightnessContrast, RandomGamma, CLAHE, ColorJitter, HueSaturation, RandomShadow |
| **노이즈/블러** | GaussNoise, ISONoise, MotionBlur, GaussianBlur, Sharpen |
| **차단** | CoarseDropout (1~8 holes, 8~48px) |
| **혼합** | Mixup (alpha=0.4, p=0.5), CutMix (alpha=0.4, p=0.3) — 물리적 일관성을 위해 모든 뷰에 동일 패치 적용 |

### 3.3 학습 세부 설정

| 항목 | 값 |
|------|-----|
| Optimizer | AdamW (LR=2e-4, weight_decay=1e-4) |
| Scheduler | CosineAnnealingLR |
| Batch Size | 2 (Gradient Accumulation 8 → 유효 BS=16) |
| Precision | bfloat16 (AMP) |
| Image Size | 512x512 |
| Fold Collapse 대응 | ep20까지 val>=0.5 시 자동 재시도 (최대 2회) |
| 재현성 | `torch.backends.cudnn.deterministic=True`, 고정 seed |

---

## 4. 추론 및 후처리

### 4.1 Test-Time Augmentation (TTA)
- **8회 추론**: 1회 clean + 7회 랜덤 augmentation (HorizontalFlip, Affine, BrightnessContrast, Perspective, GridDistortion)
- **앙상블**: Log-space geometric mean → renormalize

### 4.2 Temperature Scaling
- 모델 출력 logit을 temperature T로 나누어 확률 calibration
- **핵심 발견**: Dev/OOF 기반 T 최적화는 무의미 (모델이 train/dev에서 100% 정확 → T 구별 불가)
- T는 **제출 피드백(Public Score)으로만** 결정
- `log_odds / T → sigmoid → 제출`

### 4.3 Multi-Seed Ensemble (최종 제출)

동일 아키텍처 + 동일 하이퍼파라미터, SEED만 변경하여 2개 모델을 학습한 뒤 기하평균 앙상블:

| 버전 | SEED | Phase1 ep/patience | Phase2 ep/patience | Phase1 OOF | 단독 최적 T | 단독 Public Score |
|------|------|-------------------|-------------------|-----------|-----------|-----------------|
| **v12** | 47 | 100/35 | 60/20 | 0.003546 | 0.450 | 0.0036417 |
| **v14** | 49 | 100/35 | 60/20 | 0.003100 | - | 0.0132931 (T=0.5) |

- v14는 단독 성능은 부족하지만 (SEED 민감도), 앙상블에서 다양성을 제공하여 성능 향상
- **최종 앙상블**: v12(5fold) + v14(5fold) = 10개 모델의 geometric mean → T-scaling

### 4.4 Fold Collapse 및 재학습

소규모 데이터(~1,000장)에서 특정 fold가 학습 초기 val_loss >= 0.5에 고착되는 **fold collapse** 현상이 반복 발생:

- **자동 감지/재시도**: epoch 20까지 best_val >= 0.5이면 해당 fold를 자동 중단 후 재시도 (최대 2회, train.py 내장)
- **v14 Fold 3**: collapse가 3회 발생하여 수동 재학습으로 해결 (최종 best val 0.0043)
- **원인**: SEED에 따른 초기 가중치 + 데이터 분할 조합이 불안정한 gradient landscape를 형성
- **재현 시 주의**: SEED 변경 시 동일 fold에서 collapse가 발생하지 않을 수도 있고, 다른 fold에서 발생할 수도 있음. 자동 재시도 메커니즘이 이를 처리함

---

## 5. 핵심 발견 및 실험 이력

### 5.1 핵심 발견

1. **BS=2 필수**: BS=4에서 DevLL=0.2336 vs BS=2에서 DevLL=0.0687 → 1,000샘플에서 BS>=4는 수렴 실패
2. **CE Loss > Focal Loss**: FocalLoss + Mixup이 초기 gradient 불안정 유발 → fold collapse (val_loss ≈ 0.693 고착)
3. **Warmup 제거**: `--no-warmup`으로 CE Loss만 사용 시 안정적 수렴
4. **T-scaling 절벽**: T<=0.210에서 LogLoss 0.0078→0.025로 급등 → 하한선 존재
5. **Phase1 epoch/patience 최적점**: 100/35가 최적. 300/50 (v13)은 OOF 최고(0.0027)지만 test 7배 악화 → 과적합
6. **SEED 민감도**: 동일 설정 + SEED 변경만으로 test 성능 3.7배 차이 (v12 vs v14 단독)
7. **앙상블 효과**: 단독 성능 부족한 모델도 앙상블에서 다양성 기여 가능 (v12+v14 > v12 단독)

### 5.2 실험 이력 요약

| 버전 | SEED | 변경점 | Public Score | 교훈 |
|------|------|--------|-------------|------|
| v5 | 42 | ConvNeXt-Large + CE Loss + no-warmup | 0.0078247 | 기본 레시피 확립 |
| v7 | - | ConvNeXt-XLarge | 0.2038 | 오버피팅, 파라미터 과다 |
| v9 | - | BS=4 | 0.2336 | BS=2 필수 확인 |
| v11 | 46 | seed=46, 동일 레시피 | 0.0057420 | multi-seed 시작 |
| v12 | 47 | epoch/patience 증가 (100/35, 60/20) | 0.0036417 | 충분한 학습이 핵심 |
| v13 | 48 | epoch/patience 대폭 증가 (300/50, 100/30) | 0.0246 | 과적합 — epoch 상한선 확인 |
| v14 | 49 | v12 설정 동일, SEED만 변경 | 0.0133 | SEED 민감도, 앙상블용 |
| **v12+v14** | - | **2모델 기하평균 앙상블** | **0.0032386** | **최종 제출** |

---

## 6. 최종 결과

| 항목 | 값 |
|------|-----|
| **최종 Private Score** | **0.01063** |
| **Private 순위** | **3위** |
| **최종 Public Score** | **0.0032385603** |
| **Public 순위** | **2위** |
| **사용 모델** | v12 (SEED=47) + v14 (SEED=49) 앙상블 |
| **최적 Temperature** | T=0.550 |
| **앙상블 방법** | 10-fold geometric mean (v12 5fold + v14 5fold) |

### T-scaling 탐색 이력

```
v12+v14 앙상블:
  T=0.400 → 0.0036226
  T=0.500 → 0.0032691
  T=0.550 → 0.0032386 (최종 제출)
```

---

## 7. 개발 환경 및 재현 방법

### 7.1 개발 환경

| 항목 | 값 |
|------|-----|
| OS | Ubuntu (WSL2) / Linux 6.6.87 |
| GPU | NVIDIA RTX 5060 Ti 16GB |
| Python | 3.11 |
| PyTorch | 2.10.0 (bfloat16 AMP) |
| timm | 1.0.25 |
| albumentations | 2.0.8 |
| scikit-learn | 1.8.0 |
| scipy | 1.17.1 |
| CUDA | 12.x |

### 7.2 재현 순서

```bash
# 1. 환경 설정
pip install -r requirements.txt

# 2. 데이터 배치 (상대 경로)
# data/train/, data/dev/, data/test/, data/train.csv, data/dev.csv, data/sample_submission.csv

# 3. 영상 피처 추출 (1회)
python preprocess_video.py

# 4. Soft Label 생성 (선택, train_video_teacher.py 실행 후 data/soft_labels.json 생성)

# === v12 학습 (SEED=47) ===
# 5a. config.py에서 SEED=47 확인 후 실행
HF_HUB_OFFLINE=1 python -u train.py --no-warmup --no-focal 2>&1 | tee logs/v12.log

# === v14 학습 (SEED=49) ===
# 5b. config.py에서 SEED=49로 변경 후 실행
HF_HUB_OFFLINE=1 python -u train.py --no-warmup --no-focal 2>&1 | tee logs/v14.log

# 6. 앙상블 추론 (config.py EXTRA_CKPT_DIRS에 v14 체크포인트 경로 설정)
python inference.py

# 7. Temperature Scaling 적용 (T=0.550)
python temperature_scale.py --T 0.550 --sub submissions/<raw_submission>.csv

# 8. 제출 파일 확인
# submissions/submission_T0.550_<timestamp>.csv
```

### 7.3 파일 구조

```
structural_stability/
├── config.py                 # 하이퍼파라미터 중앙 관리
├── model.py                  # ConvNeXt-Large + CrossView Attention + GeM
├── dataset.py                # Dataset, Augmentation, Mixup/CutMix
├── train.py                  # 2-Phase 학습 파이프라인
├── inference.py              # TTA 추론 + fold 앙상블
├── temperature_scale.py      # T-scaling 적용
├── preprocess_video.py       # 영상 물리 피처 추출
├── train_video_teacher.py    # Soft label 생성용 video teacher
├── utils.py                  # 유틸리티 함수
├── requirements.txt          # 라이브러리 버전
├── data/
│   ├── train/                # 학습 데이터 (1,000 샘플)
│   ├── dev/                  # 개발 데이터 (100 샘플)
│   ├── test/                 # 평가 데이터 (1,000 샘플)
│   ├── train.csv
│   ├── dev.csv
│   ├── sample_submission.csv
│   ├── video_features.json   # 사전 추출 영상 피처
│   └── soft_labels.json      # Video teacher soft labels
├── checkpoints/
│   ├── 20260319_2248/        # v12 (SEED=47) phase2 체크포인트
│   └── 20260325_2212/        # v14 (SEED=49) phase2 체크포인트
└── submissions/              # 제출 파일
```


---
## requirements.txt
```
torch==2.10.0
torchvision==0.25.0
timm==1.0.25
albumentations==2.0.8
opencv-python==4.13.0.86
pandas==3.0.1
numpy==2.4.2
scikit-learn==1.8.0
scipy==1.17.1
matplotlib>=3.7.0
```


---
## config.py — 하이퍼파라미터 중앙 관리


In [ ]:
import os
from pathlib import Path

class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    ROOT        = Path(__file__).parent
    DATA_DIR    = ROOT / 'data'
    TRAIN_DIR   = DATA_DIR / 'train'
    DEV_DIR     = DATA_DIR / 'dev'
    TEST_DIR    = DATA_DIR / 'test'
    TRAIN_CSV   = DATA_DIR / 'train.csv'
    DEV_CSV     = DATA_DIR / 'dev.csv'
    SAMPLE_SUB  = DATA_DIR / 'sample_submission.csv'
    CKPT_BASE_DIR = ROOT / 'checkpoints'
    CKPT_DIR      = ROOT / 'checkpoints'   # train.py에서 run 폴더로 덮어씌워짐
    SUBMIT_DIR    = ROOT / 'submissions'
    VIDEO_CACHE = DATA_DIR / 'video_features.json'

    # ── Image layout ───────────────────────────────────────────────────────
    IMG_EXTENSIONS = ['.png', '.jpg', '.jpeg']
    NUM_VIEWS      = 2

    # ── Model ──────────────────────────────────────────────────────────────
    BACKBONE    = 'convnext_large.fb_in22k_ft_in1k'
    IMAGE_SIZE  = 512                # 384 → 512 (BATCH_SIZE=4, GRAD_ACCUM=4로 VRAM 확보)
    NUM_CLASSES = 2
    DROP_RATE   = 0.5                # 0.3 → 0.5 (소규모 데이터 정규화 강화)
    DROP_PATH   = 0.3
    AUX_WEIGHT  = 0.3               # 보조 손실(영상 운동량 회귀) 가중치

    # ── Video Features ─────────────────────────────────────────────────────────
    VIDEO_FEAT_DIM  = 8             # head에 직접 concat하는 영상 피처 차원
    VIDEO_FEAT_DROP = 0.5           # 학습 시 영상 피처 per-sample masking 확률 (test 분포 맞춤)

    # ── Pseudo Labeling ────────────────────────────────────────────────────────
    PSEUDO_CONF = 0.80

    # ── Soft Label (Video Teacher KD) ─────────────────────────────────────────
    SOFT_LABEL_PATH   = DATA_DIR / 'soft_labels.json'
    SOFT_LABEL_WEIGHT = 0.7

    # ── Phase 2 Validation ────────────────────────────────────────────────────
    PHASE2_VAL_RATIO = 0.1          # combined data 중 validation으로 사용할 비율
    PHASE2_PATIENCE  = 20

    # ── Heterogeneous Ensemble ─────────────────────────────────────────────────
    EXTRA_CKPT_DIRS = ['checkpoints/20260325_2212']  # v14 (v12+v14 앙상블)

    # ── Training ───────────────────────────────────────────────────────────
    SEED        = 47                   # v12: SEED=47, v14: SEED=49
    N_FOLDS     = 5
    EPOCHS      = 50
    BATCH_SIZE  = 2
    GRAD_ACCUM  = 8                  # 유효 BS=16
    NUM_WORKERS = 2

    # Optimizer
    LR           = 2e-4
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 1e-4

    # Loss
    LABEL_SMOOTHING = 0.0   # CE Loss 사용 (--no-focal)

    # SAM (Sharpness-Aware Minimization)
    USE_SAM  = False                 # SAM 폐기 (AMP 호환 불가)
    SAM_RHO  = 0.05

    # EMA (Exponential Moving Average)
    USE_EMA   = False
    EMA_DECAY = 0.999

    # ── Mixup / CutMix ─────────────────────────────────────────────────────
    MIXUP_ALPHA = 0.4
    MIXUP_PROB  = 0.5
    CUTMIX_ALPHA = 0.4
    CUTMIX_PROB  = 0.3

    # ── Domain Adaptation ──────────────────────────────────────────────────
    PHASE1_EPOCHS      = 100
    PHASE1_PATIENCE    = 35
    PHASE1_COSINE_TMAX = 100
    PHASE2_EPOCHS      = 60
    PHASE2_COSINE_TMAX = 60       # Phase2 cosine 스케줄 길이
    PHASE2_LR       = 5e-5

    # ── Fold 자동 재시도 ─────────────────────────────────────────────────
    FOLD_QUALITY_THRESHOLD  = 0.02   # 이하면 합격
    FOLD_COLLAPSE_THRESHOLD = 0.5    # 이상이면 collapse 판정
    FOLD_COLLAPSE_CHECK_EPOCH = 20   # 이 epoch까지 best_val > 0.5면 즉시 중단
    FOLD_MAX_RETRIES = 2             # fold당 최대 재시도 횟수

    # ── Inference ──────────────────────────────────────────────────────────
    TTA_TIMES    = 8
    USE_ALL_FOLD = True

    # ── Misc ───────────────────────────────────────────────────────────────
    AMP    = True
    DEVICE = 'cuda'

CFG = Config()

CFG.CKPT_BASE_DIR.mkdir(parents=True, exist_ok=True)
CFG.SUBMIT_DIR.mkdir(parents=True, exist_ok=True)


---
## model.py — ConvNeXt-Large + CrossView Attention + GeM


In [ ]:
"""
Multi-View Dual-Stream + Cross-View Attention 모델 (Option B)

변경사항:
  - Backbone: EfficientNet-B4 → ConvNeXt-Large (1536ch)
  - Aux Head: 영상 운동량(motion_score) 회귀 → 이미지에서 물리 운동 예측 학습
    (Knowledge Distillation from video to image model)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from config import CFG


# ── Cross-View Attention ─────────────────────────────────────────────────────

class CrossViewAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.attn_a2b = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_b2a = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_a   = nn.LayerNorm(dim)
        self.norm_b   = nn.LayerNorm(dim)
        self.ffn_a    = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 2, dim)
        )
        self.ffn_b    = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 2, dim)
        )
        self.norm_a2  = nn.LayerNorm(dim)
        self.norm_b2  = nn.LayerNorm(dim)

    def forward(self, feat_a: torch.Tensor, feat_b: torch.Tensor):
        # Cross-attention
        ctx_a, _ = self.attn_a2b(query=feat_a, key=feat_b, value=feat_b)
        feat_a   = self.norm_a(feat_a + ctx_a)
        ctx_b, _ = self.attn_b2a(query=feat_b, key=feat_a, value=feat_a)
        feat_b   = self.norm_b(feat_b + ctx_b)
        # FFN
        feat_a = self.norm_a2(feat_a + self.ffn_a(feat_a))
        feat_b = self.norm_b2(feat_b + self.ffn_b(feat_b))
        return feat_a, feat_b


# ── GeM Pooling ──────────────────────────────────────────────────────────────

class GeM(nn.Module):
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            kernel_size=(x.size(-2), x.size(-1))
        ).pow(1.0 / self.p).flatten(1)


# ── Main Model ───────────────────────────────────────────────────────────────

class StructureModel(nn.Module):
    """
    forward(views, training=False)
      training=True  → (logits, motion_pred)  학습 시
      training=False → logits                 추론 시
    """

    def __init__(
        self,
        backbone_name:  str   = CFG.BACKBONE,
        num_classes:    int   = CFG.NUM_CLASSES,
        drop_rate:      float = CFG.DROP_RATE,
        drop_path:      float = CFG.DROP_PATH,
        video_feat_dim: int   = CFG.VIDEO_FEAT_DIM,
        pretrained:     bool  = True,
    ):
        super().__init__()

        # ── Backbone ─────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            features_only=True,
            drop_rate=drop_rate,
            drop_path_rate=drop_path,
        )
        feat_channels = self.backbone.feature_info.channels()
        self.feat_dim = feat_channels[-1]   # ConvNeXt-Large = 1536

        # ── Projection ───────────────────────────────────────────────────
        attn_dim = 512
        self.proj = nn.Sequential(
            nn.Conv2d(self.feat_dim, attn_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(attn_dim),
            nn.GELU(),
            nn.Dropout2d(p=0.1),   # Spatial Dropout 추가
        )

        # ── Cross-View Attention ─────────────────────────────────────────
        self.cross_attn = CrossViewAttention(dim=attn_dim, num_heads=8, dropout=0.15)

        # ── Pooling ──────────────────────────────────────────────────────
        self.pool = GeM(p=3.0)

        fused_dim = attn_dim * 2   # 두 뷰 concat
        self.video_feat_dim = video_feat_dim
        head_in_dim = fused_dim + video_feat_dim   # 영상 피처 직접 concat

        # ── 분류 헤드 ─────────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(head_in_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(drop_rate * 0.5),
            nn.Linear(256, num_classes),
        )

        # ── 보조 헤드: 영상 운동량(motion_score_norm) 회귀 ─────────────────
        # 이미지만으로 구조물의 물리적 운동량을 예측하도록 강제 학습
        # → 경계(Boundary) 케이스 판별 능력 향상
        self.aux_head = nn.Sequential(
            nn.Linear(fused_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid(),   # [0, 1] 정규화된 운동량
        )

        self._init_weights()

    def _init_weights(self):
        for m in list(self.head.modules()) + list(self.aux_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode_view(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        f = feats[-1]
        # Swin Transformer: features_only output is (B, H, W, C) → convert to (B, C, H, W)
        if f.ndim == 4 and f.shape[1] == f.shape[2]:
            f = f.permute(0, 3, 1, 2).contiguous()
        return self.proj(f)           # (B, attn_dim, H', W')

    def get_fused(self, views: torch.Tensor) -> torch.Tensor:
        B, V, C, H, W = views.shape
        x_flat  = views.view(B * V, C, H, W)
        f_flat  = self.encode_view(x_flat)
        _, D, Hf, Wf = f_flat.shape
        f_split = f_flat.view(B, V, D, Hf, Wf)

        feat_a = f_split[:, 0]
        feat_b = f_split[:, 1]

        def to_seq(t):
            return t.flatten(2).permute(0, 2, 1)   # (B, N, D)

        def to_pool(seq):
            t = seq.permute(0, 2, 1).view(B, -1, Hf, Wf)
            return self.pool(t)                     # (B, D)

        seq_a, seq_b = self.cross_attn(to_seq(feat_a), to_seq(feat_b))
        return torch.cat([to_pool(seq_a), to_pool(seq_b)], dim=1)   # (B, 2D)

    def forward(self, views: torch.Tensor, video_feats=None, training: bool = False):
        fused  = self.get_fused(views)
        if video_feats is None:
            video_feats = torch.zeros(fused.size(0), self.video_feat_dim, device=fused.device)
        head_input = torch.cat([fused, video_feats], dim=1)
        logits = self.head(head_input)
        if training:
            motion_pred = self.aux_head(fused)   # (B, 1)
            return logits, motion_pred
        return logits


# ── 빌더 ─────────────────────────────────────────────────────────────────────

def build_model(pretrained: bool = True) -> StructureModel:
    return StructureModel(pretrained=pretrained)


if __name__ == '__main__':
    model = build_model(pretrained=False)
    dummy = torch.randn(2, 2, 3, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)
    logits, motion = model(dummy, training=True)
    print(f'Input : {dummy.shape}')
    print(f'Logits: {logits.shape}')
    print(f'Motion: {motion.shape}')
    total = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Params: {total:.1f}M')


---
## dataset.py — Dataset, Augmentation, Mixup/CutMix


In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

from config import CFG

# 영상 피처 키 목록 (preprocess_video.py와 순서 일치해야 함)
VIDEO_FEAT_KEYS = [
    'motion_score_norm', 'frame_diff_norm', 'max_flow_norm',
    'dominant_freq_norm', 'motion_accel_norm', 'collapse_frame_norm',
    'top_motion_norm', 'bottom_motion_norm',
]


# ── 이미지 로더 ────────────────────────────────────────────────────────────────

def load_views(sample_dir: Path, num_views: int = 2) -> list[np.ndarray]:
    """front.png(정면), top.png(위) 순서로 로드."""
    view_names = ['front.png', 'top.png']
    imgs = []
    for name in view_names[:num_views]:
        path = sample_dir / name
        img  = cv2.imread(str(path))
        if img is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없습니다: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        imgs.append(img)
    return imgs


# ── 영상 특징 캐시 로드 ────────────────────────────────────────────────────────

def load_video_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)


# ── Augmentation ──────────────────────────────────────────────────────────────

def get_train_transform(image_size: int):
    return A.Compose([
        A.Resize(image_size, image_size),

        # 공간 변환
        A.Affine(
            translate_percent=(-0.1, 0.1), scale=(0.85, 1.15), rotate=(-15, 15), p=0.7
        ),
        A.Perspective(scale=(0.05, 0.15), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, p=0.2),

        # 광원/색상 변화
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4),
            A.RandomGamma(gamma_limit=(60, 140)),
            A.CLAHE(clip_limit=4.0),
        ], p=0.8),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=30, p=0.5),
        A.RandomShadow(p=0.3),

        # 노이즈 & 블러
        A.OneOf([
            A.GaussNoise(),
            A.ISONoise(color_shift=(0.01, 0.05)),
        ], p=0.4),
        A.OneOf([
            A.MotionBlur(blur_limit=5),
            A.GaussianBlur(blur_limit=5),
            A.Sharpen(alpha=(0.2, 0.5), p=1.0),
        ], p=0.3),

        # 차단
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(8, 48),
            hole_width_range=(8, 48),
            p=0.35
        ),

        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transform(image_size: int):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_tta_transform(image_size: int):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.OneOf([
            A.HorizontalFlip(p=1.0),
            A.Affine(translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-10, 10), p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.Perspective(scale=(0.03, 0.08), p=1.0),
            A.GridDistortion(num_steps=3, distort_limit=0.2, p=1.0),
        ], p=0.9),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


# ── Dataset ───────────────────────────────────────────────────────────────────

class StructureDataset(Dataset):
    LABEL2IDX = {'stable': 0, 'unstable': 1}

    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        transform=None,
        is_test: bool = False,
        video_cache: dict = None,
        soft_labels: dict = None,
    ):
        self.df          = df.reset_index(drop=True)
        self.data_dir    = Path(data_dir)
        self.transform   = transform
        self.is_test     = is_test
        self.video_cache = video_cache or {}
        self.soft_labels = soft_labels or {}   # {sample_id: [p_stable, p_unstable]}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        sample_id  = row['id']
        sample_dir = self.data_dir / sample_id

        views = load_views(sample_dir, num_views=CFG.NUM_VIEWS)

        tensors = []
        for img in views:
            aug = self.transform(image=img)['image'] if self.transform else \
                  torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
            tensors.append(aug)
        views_tensor = torch.stack(tensors, dim=0)   # (V, C, H, W)

        # 영상 피처 (train만 존재, 없으면 0 — test/pseudo 포함)
        vfeat = self.video_cache.get(sample_id, {})
        video_feats = torch.tensor(
            [vfeat.get(k, 0.0) for k in VIDEO_FEAT_KEYS], dtype=torch.float32
        )  # (VIDEO_FEAT_DIM,)

        if self.is_test:
            return views_tensor, sample_id

        label = self.LABEL2IDX[row['label'].lower()]

        # soft label: [p_stable, p_unstable], 없으면 sentinel [-1, -1]
        if sample_id in self.soft_labels:
            soft = torch.tensor(self.soft_labels[sample_id], dtype=torch.float32)
        else:
            soft = torch.tensor([-1.0, -1.0], dtype=torch.float32)

        return views_tensor, torch.tensor(label, dtype=torch.long), video_feats, soft


# ── Mixup / CutMix ────────────────────────────────────────────────────────────

def mixup_data(views, labels, video_feats, alpha: float = CFG.MIXUP_ALPHA):
    """
    views      : (B, V, C, H, W)
    labels     : (B,) LongTensor
    video_feats: (B, VIDEO_FEAT_DIM) FloatTensor
    """
    lam   = np.random.beta(alpha, alpha)
    index = torch.randperm(views.size(0), device=views.device)
    mixed_views      = lam * views + (1 - lam) * views[index]
    mixed_video_feats = lam * video_feats + (1 - lam) * video_feats[index]
    return mixed_views, labels, labels[index], mixed_video_feats, lam


def cutmix_data(views, labels, video_feats, alpha: float = CFG.CUTMIX_ALPHA):
    """
    같은 패치를 모든 뷰에 적용하여 물리적 일관성 유지.
    """
    lam   = np.random.beta(alpha, alpha)
    B, V, C, H, W = views.shape
    index = torch.randperm(B, device=views.device)

    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0);  x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0);  y2 = min(cy + cut_h // 2, H)

    mixed_views = views.clone()
    mixed_views[:, :, :, y1:y2, x1:x2] = views[index, :, :, y1:y2, x1:x2]

    lam_actual    = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)
    mixed_video_feats = lam_actual * video_feats + (1 - lam_actual) * video_feats[index]
    return mixed_views, labels, labels[index], mixed_video_feats, lam_actual


# ── 헬퍼 ──────────────────────────────────────────────────────────────────────

def make_combined_df(train_csv: Path, dev_csv: Path) -> pd.DataFrame:
    train_df = pd.read_csv(train_csv)
    dev_df   = pd.read_csv(dev_csv)
    train_df['source'] = 'train'
    dev_df['source']   = 'dev'
    return pd.concat([train_df, dev_df], ignore_index=True)


---
## preprocess_video.py — 영상 물리 피처 추출


In [ ]:
"""
train/simulation.mp4 에서 물리 운동 특징 추출 → data/video_features.json 저장
학습 전 딱 1회만 실행: python preprocess_video.py

추출 특징 (8개):
  motion_score   : Farneback Optical Flow 평균 이동량 (불안정할수록 큼)
  frame_diff     : 첫 프레임 vs 마지막 프레임 평균 절대 차이
  max_flow       : 프레임당 최대 픽셀 이동량 최댓값
  dominant_freq  : 운동량 시계열의 지배 주파수 (FFT — 진동 주기)
  motion_accel   : 운동량 변화율 평균 (가속도/jerk — 급격한 변화 = 불안정)
  collapse_frame : 최대 운동 발생 정규화 시점 (0=초반 붕괴, 1=말미 붕괴)
  top_motion     : 이미지 상단 절반 평균 flow (상부 구조 움직임)
  bottom_motion  : 이미지 하단 절반 평균 flow (하부/기반 움직임)

저장 후 min-max 정규화해서 [0, 1] 범위로 변환.
"""

import cv2
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

TRAIN_DIR  = Path(__file__).parent / 'data' / 'train'
OUT_PATH   = Path(__file__).parent / 'data' / 'video_features.json'
MAX_FRAMES = 60   # 최대 60프레임(=10초 @ 6fps)
N_WORKERS  = 4

RAW_KEYS = [
    'motion_score', 'frame_diff', 'max_flow',
    'dominant_freq', 'motion_accel', 'collapse_frame',
    'top_motion', 'bottom_motion',
]


# ── 단일 영상 처리 ─────────────────────────────────────────────────────────────

def extract_features(sample_dir: Path) -> dict:
    video_path = sample_dir / 'simulation.mp4'
    zero = {k: 0.0 for k in RAW_KEYS}
    if not video_path.exists():
        return zero

    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while len(frames) < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(gray)
    cap.release()

    if len(frames) < 2:
        return zero

    H = frames[0].shape[0]
    half = H // 2

    # ── Optical Flow ──────────────────────────────────────────────────────────
    motion_scores, max_flows = [], []
    top_motions, bottom_motions = [], []

    for i in range(1, len(frames)):
        flow = cv2.calcOpticalFlowFarneback(
            frames[i-1], frames[i], None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )
        mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
        motion_scores.append(float(mag.mean()))
        max_flows.append(float(mag.max()))
        top_motions.append(float(mag[:half].mean()))
        bottom_motions.append(float(mag[half:].mean()))

    # ── 첫 프레임 vs 마지막 프레임 차이 ─────────────────────────────────────────
    frame_diff = float(np.abs(frames[-1].astype(float) - frames[0].astype(float)).mean())

    # ── 지배 주파수 (FFT) ────────────────────────────────────────────────────
    if len(motion_scores) >= 4:
        fft  = np.abs(np.fft.rfft(motion_scores))
        freq = np.fft.rfftfreq(len(motion_scores))
        dominant_freq = float(freq[np.argmax(fft[1:]) + 1]) if len(fft) > 1 else 0.0
    else:
        dominant_freq = 0.0

    # ── 가속도 (jerk: 운동량 변화율) ─────────────────────────────────────────
    motion_accel = float(np.mean(np.abs(np.diff(motion_scores)))) if len(motion_scores) >= 2 else 0.0

    # ── 붕괴 시점 (최대 운동 발생 정규화 시점) ──────────────────────────────────
    collapse_frame = float(np.argmax(motion_scores) / len(motion_scores)) if motion_scores else 0.0

    return {
        'motion_score'  : float(np.mean(motion_scores)),
        'frame_diff'    : frame_diff,
        'max_flow'      : float(np.max(max_flows)),
        'dominant_freq' : dominant_freq,
        'motion_accel'  : motion_accel,
        'collapse_frame': collapse_frame,
        'top_motion'    : float(np.mean(top_motions)),
        'bottom_motion' : float(np.mean(bottom_motions)),
    }


def process_one(sample_dir: Path) -> tuple[str, dict]:
    return sample_dir.name, extract_features(sample_dir)


# ── 전체 처리 ─────────────────────────────────────────────────────────────────

def main():
    sample_dirs = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
    print(f'처리할 샘플: {len(sample_dirs)}개')

    raw = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_one, d): d for d in sample_dirs}
        for future in tqdm(as_completed(futures), total=len(futures), desc='영상 처리'):
            name, feats = future.result()
            raw[name] = feats

    # ── Min-Max 정규화 ────────────────────────────────────────────────────────
    for key in RAW_KEYS:
        vals  = np.array([v[key] for v in raw.values()])
        vmin, vmax = vals.min(), vals.max()
        denom = vmax - vmin if vmax > vmin else 1.0
        for name in raw:
            raw[name][f'{key}_norm'] = float((raw[name][key] - vmin) / denom)

    # ── 요약 출력 ─────────────────────────────────────────────────────────────
    for key in RAW_KEYS:
        norms = [v[f'{key}_norm'] for v in raw.values()]
        print(f'  [{key}_norm] min={min(norms):.3f}  max={max(norms):.3f}  mean={np.mean(norms):.3f}')

    with open(OUT_PATH, 'w') as f:
        json.dump(raw, f, indent=2)
    print(f'\n저장 완료 → {OUT_PATH}')


if __name__ == '__main__':
    main()


---
## train_video_teacher.py — Soft label 생성용 video teacher


In [ ]:
"""
Video Teacher 모델 학습 (Step 2: Soft Label KD)

simulation.mp4의 마지막 프레임으로 Teacher 분류기 학습.
→ OOF soft labels (train) + 앙상블 soft labels (dev) 생성
→ data/soft_labels.json 저장

실행:
    python train_video_teacher.py

출력:
    data/soft_labels.json  ← train.py가 자동으로 로드
"""

import cv2
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import albumentations as A
from albumentations.pytorch import ToTensorV2

from config import CFG

# ── 하이퍼파라미터 ─────────────────────────────────────────────────────────────
VT_BACKBONE  = 'efficientnet_b0'    # 경량 모델 (video는 신호가 강해서 작은 모델로 충분)
VT_IMAGE_SIZE = 384
VT_EPOCHS     = 40
VT_LR         = 3e-4
VT_BATCH      = 16
VT_N_FOLDS    = 5
VT_N_FRAMES   = 5                   # 마지막 N 프레임 평균

# ── 마지막 프레임 추출 ─────────────────────────────────────────────────────────

def extract_last_frames(video_path: Path, n_frames: int = VT_N_FRAMES) -> np.ndarray:
    """simulation.mp4에서 마지막 n_frames 프레임을 평균낸 이미지 반환 (H, W, 3)."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    start_frame = max(0, total - n_frames)
    for i in range(start_frame, total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    return np.mean(frames, axis=0).astype(np.uint8)


# ── Dataset ───────────────────────────────────────────────────────────────────

def get_transform(is_train: bool):
    if is_train:
        return A.Compose([
            A.Resize(VT_IMAGE_SIZE, VT_IMAGE_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.7),
            A.Affine(translate_percent=(-0.1, 0.1), scale=(0.85, 1.15), rotate=(-15, 15), p=0.6),
            A.OneOf([A.GaussNoise(), A.GaussianBlur(blur_limit=3)], p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(VT_IMAGE_SIZE, VT_IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


class VideoFrameDataset(Dataset):
    LABEL2IDX = {'stable': 0, 'unstable': 1}

    def __init__(self, df: pd.DataFrame, data_dir: Path, transform=None, is_test: bool = False):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = Path(data_dir)
        self.transform = transform
        self.is_test   = is_test
        self._cache    = {}   # 메모리 캐시 (프레임 추출 비용 절약)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        sample_id = row['id']

        if sample_id not in self._cache:
            video_path = self.data_dir / sample_id / 'simulation.mp4'
            self._cache[sample_id] = extract_last_frames(video_path)
        img = self._cache[sample_id]

        if self.transform:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0

        if self.is_test:
            return img, sample_id

        label = self.LABEL2IDX[row['label'].lower()]
        return img, torch.tensor(label, dtype=torch.long)


# ── Model ─────────────────────────────────────────────────────────────────────

def build_video_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(VT_BACKBONE, pretrained=pretrained, num_classes=2, drop_rate=0.3)
    return model


# ── Training ──────────────────────────────────────────────────────────────────

def train_vt_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    criterion  = nn.CrossEntropyLoss(label_smoothing=0.05)
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast('cuda', enabled=True):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def val_vt_epoch(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast('cuda', enabled=True):
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return log_loss(labels, probs), probs


@torch.no_grad()
def predict_vt(model, loader, device):
    model.eval()
    all_probs = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        with autocast('cuda', enabled=True):
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


# ── 메인 ──────────────────────────────────────────────────────────────────────

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    print(f'Video Teacher: {VT_BACKBONE} | Epochs: {VT_EPOCHS} | Folds: {VT_N_FOLDS}')

    train_df = pd.read_csv(CFG.TRAIN_CSV)
    dev_df   = pd.read_csv(CFG.DEV_CSV)

    # OOF 예측값 저장 배열 (train용)
    oof_preds = np.zeros((len(train_df), 2))

    skf = StratifiedKFold(n_splits=VT_N_FOLDS, shuffle=True, random_state=CFG.SEED)

    # dev 앙상블 예측 (모든 fold 평균)
    dev_fold_preds = []

    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
        print(f'\n── Fold {fold+1}/{VT_N_FOLDS} ──')
        trn_df = train_df.iloc[trn_idx]
        val_df = train_df.iloc[val_idx]

        trn_ds = VideoFrameDataset(trn_df, CFG.TRAIN_DIR, get_transform(True))
        val_ds = VideoFrameDataset(val_df, CFG.TRAIN_DIR, get_transform(False))
        dev_ds = VideoFrameDataset(dev_df, CFG.TRAIN_DIR, get_transform(False))  # dev에는 video 없음

        trn_loader = DataLoader(trn_ds, batch_size=VT_BATCH, shuffle=True,  num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=VT_BATCH * 2, shuffle=False, num_workers=4, pin_memory=True)

        model     = build_video_model(pretrained=True).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=VT_LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=VT_EPOCHS, eta_min=1e-6)
        scaler    = GradScaler('cuda')

        best_val  = float('inf')
        best_preds = None
        ckpt_path = CFG.CKPT_BASE_DIR / f'video_teacher_fold{fold+1}.pth'

        for epoch in range(1, VT_EPOCHS + 1):
            t0       = time.time()
            trn_loss = train_vt_epoch(model, trn_loader, optimizer, scaler, device)
            val_loss, val_preds = val_vt_epoch(model, val_loader, device)
            scheduler.step()

            mark = ' ★' if val_loss < best_val else ''
            print(f'  Epoch {epoch:02d}/{VT_EPOCHS} | Train: {trn_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.1f}s{mark}', flush=True)

            if val_loss < best_val:
                best_val   = val_loss
                best_preds = val_preds
                torch.save(model.state_dict(), ckpt_path)

        # OOF 저장
        oof_preds[val_idx] = best_preds
        oof_logloss = log_loss(train_df.iloc[val_idx]['label'].map({'stable': 0, 'unstable': 1}), best_preds)
        print(f'  Fold {fold+1} OOF LogLoss: {oof_logloss:.4f}')

        del model, trn_ds, val_ds
        torch.cuda.empty_cache()

    # 전체 OOF LogLoss
    all_labels = train_df['label'].map({'stable': 0, 'unstable': 1}).values
    total_oof  = log_loss(all_labels, oof_preds)
    print(f'\nVideo Teacher OOF LogLoss (전체): {total_oof:.4f}')

    # dev 앙상블 예측 (각 fold 모델로 예측)
    print('\nDev set 앙상블 예측 중...')
    dev_preds_list = []
    for fold in range(VT_N_FOLDS):
        ckpt_path = CFG.CKPT_BASE_DIR / f'video_teacher_fold{fold+1}.pth'
        if not ckpt_path.exists():
            continue
        model = build_video_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        dev_ds     = VideoFrameDataset(dev_df, CFG.TRAIN_DIR, get_transform(False))
        dev_loader = DataLoader(dev_ds, batch_size=VT_BATCH * 2, shuffle=False, num_workers=4, pin_memory=True)

        # dev에는 simulation.mp4 없음 → VideoFrameDataset이 파일 없으면 에러
        # dev 디렉토리에서 영상 없는 경우를 처리 (simulation.mp4 존재 여부 확인)
        dev_has_video = (CFG.DEV_DIR / dev_df.iloc[0]['id'] / 'simulation.mp4').exists()
        if not dev_has_video:
            print(f'  Dev set에 simulation.mp4 없음 → dev soft labels 생성 불가')
            break

        preds = predict_vt(model, dev_loader, device)
        dev_preds_list.append(preds)
        del model; torch.cuda.empty_cache()

    # ── soft_labels.json 저장 ─────────────────────────────────────────────────
    soft_labels = {}

    # train OOF predictions
    for i, row in train_df.iterrows():
        soft_labels[row['id']] = oof_preds[i].tolist()   # [p_stable, p_unstable]

    # dev predictions (앙상블 가능한 경우만)
    if dev_preds_list:
        dev_ensemble = np.exp(np.mean([np.log(p + 1e-8) for p in dev_preds_list], axis=0))
        dev_ensemble = dev_ensemble / dev_ensemble.sum(axis=1, keepdims=True)
        for i, row in dev_df.iterrows():
            soft_labels[row['id']] = dev_ensemble[i].tolist()
        print(f'Dev soft labels 저장: {len(dev_df)}개')

    out_path = CFG.DATA_DIR / 'soft_labels.json'
    with open(out_path, 'w') as f:
        json.dump(soft_labels, f, indent=2)
    print(f'\nSoft labels 저장 완료: {out_path}')
    print(f'  총 {len(soft_labels)}개 샘플')


if __name__ == '__main__':
    main()


---
## train.py — 2-Phase 학습 파이프라인


In [ ]:
"""
학습 파이프라인 v6

개선사항 (v5 대비):
  - SAM (Sharpness-Aware Minimization): 평탄한 minimum → 일반화 개선
  - EMA (Exponential Moving Average): 가중치 지수이동평균 → 노이즈 감소
  - OOF 예측 저장: Phase1 held-out 예측으로 T 최적화 (dev 편향 해소)
"""

import copy
import datetime
import gc
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset
from torch.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from config import CFG
from dataset import (
    StructureDataset, get_train_transform, get_val_transform,
    make_combined_df, mixup_data, cutmix_data, load_video_cache
)
from pathlib import Path
from model import build_model


# ── EMA ──────────────────────────────────────────────────────────────────────

class ModelEMA:
    """Exponential Moving Average of model parameters."""

    def __init__(self, model, decay=0.999):
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

    def state_dict(self):
        return self.module.state_dict()


# ── SAM helpers ──────────────────────────────────────────────────────────────

@torch.no_grad()
def _sam_perturb(model, rho: float):
    """Compute SAM perturbation from current gradients and apply it."""
    grad_norm = torch.norm(
        torch.stack([p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]),
        p=2,
    )
    for p in model.parameters():
        if p.grad is None:
            continue
        e_w = rho * p.grad / (grad_norm + 1e-12)
        p.add_(e_w)
        p._sam_e_w = e_w


@torch.no_grad()
def _sam_restore(model):
    """Remove SAM perturbation."""
    for p in model.parameters():
        if hasattr(p, '_sam_e_w'):
            p.sub_(p._sam_e_w)
            del p._sam_e_w


def load_soft_labels(path: Path) -> dict:
    """soft_labels.json 로드. 없으면 빈 dict 반환."""
    if not path.exists():
        return {}
    import json
    with open(path) as f:
        data = json.load(f)
    print(f'Soft labels 로드: {len(data)}개 샘플')
    return data


# ── Run 폴더 관리 ────────────────────────────────────────────────────────────

def resolve_run_dir():
    """
    이전에 미완료된 실행이 있으면 재개, 없으면 새 타임스탬프 폴더 생성.
    checkpoints/.current_run 파일에 현재 run ID를 기록.
    """
    base   = CFG.CKPT_BASE_DIR
    marker = base / '.current_run'

    if marker.exists():
        run_id  = marker.read_text().strip()
        run_dir = base / run_id
        if run_dir.exists():
            all_p2_done = all(
                (run_dir / f'fold{i}_phase2.pth').exists()
                for i in range(1, CFG.N_FOLDS + 1)
            )
            has_resume = any(
                (run_dir / f'fold{i}_phase2_resume.pth').exists()
                for i in range(1, CFG.N_FOLDS + 1)
            )
            if not all_p2_done or has_resume:
                print(f'[Resume] 이전 실행 재개 → {run_id}')
                return run_dir

    run_id  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    run_dir = base / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    marker.write_text(run_id)
    print(f'[New Run] 새 실행 시작 → {run_id}')
    return run_dir


# ── 재현성 ────────────────────────────────────────────────────────────────────

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ── 샘플러 ────────────────────────────────────────────────────────────────────

def make_weighted_sampler(labels: list[int]) -> WeightedRandomSampler:
    counts  = np.bincount(labels)
    weights = 1.0 / counts[labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True,
    )


# ── Loss ──────────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()


def mixup_ce_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)


def aux_loss_fn(motion_pred, video_feats):
    """
    영상 피처의 첫 번째 차원(motion_score_norm)을 보조 회귀 타깃으로 사용.
    video_feats가 0인 샘플(test/pseudo)은 마스킹.
    """
    motion_target = video_feats[:, 0]   # motion_score_norm
    mask = motion_target > 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=motion_pred.device)
    return F.mse_loss(motion_pred[mask].squeeze(-1), motion_target[mask])


# ── Train Epoch ───────────────────────────────────────────────────────────────

def _forward_batch(model, views, labels, video_feats, soft_labels, criterion, device, warmup=False):
    """단일 배치 forward → (cls_loss, aux_loss) 반환. Mixup/CutMix 포함."""
    # Per-sample video feature dropout
    feat_mask = (torch.rand(video_feats.size(0), 1, device=device) > CFG.VIDEO_FEAT_DROP).float()
    video_feats = video_feats * feat_mask

    # Mixup / CutMix (warmup 중 비활성화)
    r = np.random.rand()
    if not warmup and r < CFG.MIXUP_PROB:
        views, la, lb, video_feats, lam = mixup_data(views, labels, video_feats)
        use_mix = True
    elif not warmup and r < CFG.MIXUP_PROB + CFG.CUTMIX_PROB:
        views, la, lb, video_feats, lam = cutmix_data(views, labels, video_feats)
        use_mix = True
    else:
        use_mix = False

    with autocast('cuda', dtype=torch.bfloat16, enabled=CFG.AMP):
        logits, motion_pred = model(views, video_feats=video_feats, training=True)

        if use_mix:
            cls_loss = mixup_ce_loss(criterion, logits, la, lb, lam)
        else:
            hard_loss = criterion(logits, labels)
            valid = soft_labels[:, 0] >= 0
            if valid.any() and CFG.SOFT_LABEL_WEIGHT > 0:
                sl = soft_labels[valid]
                sl = sl / sl.sum(dim=1, keepdim=True)
                kl = F.kl_div(
                    F.log_softmax(logits[valid], dim=1),
                    sl, reduction='batchmean'
                )
                cls_loss = (1 - CFG.SOFT_LABEL_WEIGHT) * hard_loss + CFG.SOFT_LABEL_WEIGHT * kl
            else:
                cls_loss = hard_loss

        aux_loss = aux_loss_fn(motion_pred, video_feats)

    return cls_loss, aux_loss


def train_epoch(model, loader, optimizer, criterion, scaler, device, ema=None, warmup=False):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    use_sam = CFG.USE_SAM

    _t_start = time.time()
    _t_data  = 0.0
    _t_prev  = time.time()

    for step, batch in enumerate(loader):
        _t_data += time.time() - _t_prev
        views, labels, video_feats, soft_labels = batch
        views, labels = views.to(device), labels.to(device)
        video_feats  = video_feats.to(device)
        soft_labels  = soft_labels.to(device)

        cls_loss, aux_loss = _forward_batch(
            model, views, labels, video_feats, soft_labels, criterion, device, warmup=warmup
        )
        loss = (cls_loss + CFG.AUX_WEIGHT * aux_loss) / CFG.GRAD_ACCUM
        scaler.scale(loss).backward()

        is_step = (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == len(loader)

        if is_step:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if use_sam:
                # ── SAM: perturb → second forward → restore → step ──
                _sam_perturb(model, CFG.SAM_RHO)
                optimizer.zero_grad()

                # Second forward: scaler와 분리 (unscale_ 이후 scaler.scale 금지)
                with autocast('cuda', dtype=torch.bfloat16, enabled=CFG.AMP):
                    logits2, _ = model(views, video_feats=video_feats, training=True)
                    loss2 = criterion(logits2, labels)
                loss2.backward()

                _sam_restore(model)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scaler.update()
            else:
                scaler.step(optimizer)
                scaler.update()

            if ema is not None:
                ema.update(model)

            optimizer.zero_grad()

        total_loss += (cls_loss.item() + CFG.AUX_WEIGHT * aux_loss.item()) * len(labels)
        _t_prev = time.time()

    _t_total = time.time() - _t_start
    print(f'  [Timing] data={_t_data:.0f}s  gpu={_t_total - _t_data:.0f}s  total={_t_total:.0f}s', flush=True)

    return total_loss / len(loader.dataset)


# ── Val Epoch ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def val_epoch(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for views, labels, _vf, _sl in loader:
        views = views.to(device)
        with autocast('cuda', dtype=torch.bfloat16):
            logits = model(views, training=False)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs,  axis=0)
    labels = np.concatenate(all_labels, axis=0)
    return log_loss(labels, probs), probs


# ── Single Fold Training ──────────────────────────────────────────────────────

def _train_single_fold(fold, trn_idx, val_idx, train_df, video_cache, device,
                        soft_labels, no_warmup, no_focal):
    """단일 fold 학습. Returns (best_loss, oof_probs, val_labels)"""
    print(f'\n── Fold {fold+1}/{CFG.N_FOLDS} ──')
    trn_df = train_df.iloc[trn_idx]
    val_df = train_df.iloc[val_idx]

    trn_ds = StructureDataset(trn_df, CFG.TRAIN_DIR, get_train_transform(CFG.IMAGE_SIZE),
                              video_cache=video_cache, soft_labels=soft_labels)
    val_ds = StructureDataset(val_df, CFG.TRAIN_DIR, get_val_transform(CFG.IMAGE_SIZE),
                              video_cache=video_cache)

    label_idx = [StructureDataset.LABEL2IDX[l.lower()] for l in trn_df['label'].tolist()]
    sampler   = make_weighted_sampler(label_idx)

    trn_loader = DataLoader(trn_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True,
                            persistent_workers=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                            num_workers=2, pin_memory=True,
                            persistent_workers=False)

    WARMUP_EPOCHS = 0 if no_warmup else 5
    model     = build_model(pretrained=True).to(device)
    ema       = ModelEMA(model, decay=CFG.EMA_DECAY) if CFG.USE_EMA else None
    focal_criterion = FocalLoss(gamma=2.0)
    ce_criterion    = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    if no_warmup:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.PHASE1_COSINE_TMAX, eta_min=CFG.MIN_LR
        )
    else:
        warmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        cosine_sched = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.PHASE1_COSINE_TMAX - WARMUP_EPOCHS, eta_min=CFG.MIN_LR
        )
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS]
        )
    scaler    = GradScaler('cuda', enabled=CFG.AMP)

    best_loss    = float('inf')
    patience_cnt = 0
    start_epoch  = 1
    ckpt_path   = CFG.CKPT_DIR / f'fold{fold+1}_phase1.pth'
    resume_path = CFG.CKPT_DIR / f'fold{fold+1}_phase1_resume.pth'

    # ── 중단된 fold 재개 ────────────────────────────────────────────────
    if resume_path.exists():
        state = torch.load(resume_path, map_location=device)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        best_loss    = state['best_loss']
        patience_cnt = state.get('patience_cnt', 0)
        start_epoch  = state['epoch'] + 1
        if ema is not None and 'ema' in state:
            ema.module.load_state_dict(state['ema'])
        print(f'  [Resume] Fold {fold+1} epoch {start_epoch}부터 재개 (best={best_loss:.4f})')

    for epoch in range(start_epoch, CFG.PHASE1_EPOCHS + 1):
        is_warmup = (not no_warmup) and (epoch <= WARMUP_EPOCHS)
        criterion = ce_criterion if (is_warmup or no_focal) else focal_criterion
        t0       = time.time()
        trn_loss = train_epoch(model, trn_loader, optimizer, criterion, scaler, device, ema=ema, warmup=is_warmup)

        # Validation: EMA 모델로 평가 (있으면)
        eval_model = ema.module if ema is not None else model
        val_loss, val_probs = val_epoch(eval_model, val_loader, device)
        if epoch <= CFG.PHASE1_COSINE_TMAX:
            scheduler.step()
        # epoch > PHASE1_COSINE_TMAX: LR = MIN_LR 유지

        mark = ' ★' if val_loss < best_loss else ''
        wu_tag = ' [WU]' if is_warmup else ''
        print(f'  Epoch {epoch:02d}/{CFG.PHASE1_EPOCHS} | '
              f'Train: {trn_loss:.4f} | Val: {val_loss:.4f} | '
              f'{time.time()-t0:.1f}s{mark}{wu_tag}', flush=True)

        if val_loss < best_loss:
            best_loss    = val_loss
            patience_cnt = 0
            # EMA 가중치 저장 (있으면), 없으면 일반 모델
            save_state = ema.state_dict() if ema is not None else model.state_dict()
            torch.save(save_state, ckpt_path)
        elif not is_warmup:
            patience_cnt += 1
            if patience_cnt >= CFG.PHASE1_PATIENCE:
                print(f'  Early stopping at epoch {epoch} (patience={CFG.PHASE1_PATIENCE})', flush=True)
                break

        # 조기 collapse 감지
        if epoch >= CFG.FOLD_COLLAPSE_CHECK_EPOCH and best_loss >= CFG.FOLD_COLLAPSE_THRESHOLD:
            print(f'  *** FOLD COLLAPSE 감지 (epoch {epoch}, best_val={best_loss:.4f} >= {CFG.FOLD_COLLAPSE_THRESHOLD}) ***')
            break

        # resume 체크포인트 저장 (매 epoch) — GC로 메모리 확보 후 저장
        gc.collect()
        torch.cuda.empty_cache()
        resume_state = {
            'epoch':        epoch,
            'model':        model.state_dict(),
            'optimizer':    optimizer.state_dict(),
            'scheduler':    scheduler.state_dict(),
            'scaler':       scaler.state_dict(),
            'best_loss':    best_loss,
            'patience_cnt': patience_cnt,
        }
        if ema is not None:
            resume_state['ema'] = ema.state_dict()
        torch.save(resume_state, resume_path)
        del resume_state
        gc.collect()

    # ── OOF 예측 수집 (best 체크포인트로) ────────────────────────────────
    if ckpt_path.exists():
        best_model = build_model(pretrained=False).to(device)
        best_model.load_state_dict(torch.load(ckpt_path, map_location=device))
        _, oof_probs = val_epoch(best_model, val_loader, device)
        del best_model
    else:
        # collapse로 best 체크포인트가 없는 경우
        oof_probs = np.full((len(val_idx), CFG.NUM_CLASSES), 1.0 / CFG.NUM_CLASSES)

    val_labels = [StructureDataset.LABEL2IDX[l.lower()] for l in val_df['label'].tolist()]

    print(f'  Best Val LogLoss: {best_loss:.4f} → {ckpt_path}', flush=True)
    resume_path.unlink(missing_ok=True)
    del model, trn_loader, val_loader
    if ema is not None:
        del ema
    gc.collect()
    torch.cuda.empty_cache()

    return best_loss, oof_probs, val_labels


# ── Phase 1: K-Fold ───────────────────────────────────────────────────────────

def train_phase1(train_df: pd.DataFrame, video_cache: dict, device: torch.device, soft_labels: dict = None, only_folds: list = None, no_warmup: bool = False, no_focal: bool = False):
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

    print(f'\n{"="*60}')
    print(f'  Phase 1 — {CFG.N_FOLDS}-Fold CV  |  Epochs: {CFG.PHASE1_EPOCHS}')
    print(f'  Backbone: {CFG.BACKBONE}  |  ImgSize: {CFG.IMAGE_SIZE}')
    print(f'  SAM: {CFG.USE_SAM} (rho={CFG.SAM_RHO})  EMA: {CFG.USE_EMA} (decay={CFG.EMA_DECAY})')
    print(f'  Mixup: {CFG.MIXUP_PROB}  CutMix: {CFG.CUTMIX_PROB}  AuxW: {CFG.AUX_WEIGHT}')
    if no_warmup or no_focal:
        flags = []
        if no_warmup: flags.append('no-warmup')
        if no_focal:  flags.append('no-focal (CE only)')
        print(f'  Flags: {", ".join(flags)}')
    else:
        print(f'  Warmup: 5 epochs (CE + no Mixup/CutMix + LR 0.01→1.0)')
    if only_folds:
        print(f'  대상 Fold: {only_folds}')
    print(f'{"="*60}')

    # OOF predictions 수집
    oof_preds  = np.zeros((len(train_df), CFG.NUM_CLASSES))
    oof_labels = np.zeros(len(train_df), dtype=int)

    for fold, (trn_idx, val_idx) in enumerate(
        skf.split(train_df, train_df['label'])
    ):
        if only_folds and (fold + 1) not in only_folds:
            print(f'\n── Fold {fold+1}/{CFG.N_FOLDS} ── [스킵]')
            continue

        for retry in range(CFG.FOLD_MAX_RETRIES + 1):
            if retry > 0:
                print(f'\n  *** FOLD {fold+1} RETRY {retry}/{CFG.FOLD_MAX_RETRIES} ***')
                (CFG.CKPT_DIR / f'fold{fold+1}_phase1.pth').unlink(missing_ok=True)
                (CFG.CKPT_DIR / f'fold{fold+1}_phase1_resume.pth').unlink(missing_ok=True)
                seed_everything(CFG.SEED + (retry * 1000) + fold)

            best_loss, fold_oof_probs, fold_val_labels = _train_single_fold(
                fold, trn_idx, val_idx, train_df, video_cache, device,
                soft_labels, no_warmup, no_focal
            )

            if best_loss <= CFG.FOLD_QUALITY_THRESHOLD:
                print(f'  Fold {fold+1} PASS (best_val={best_loss:.4f})')
                break
            elif retry < CFG.FOLD_MAX_RETRIES:
                print(f'  Fold {fold+1} FAIL (best_val={best_loss:.4f}) → 재시도')
            else:
                print(f'  Fold {fold+1} MAX RETRIES 도달. best_val={best_loss:.4f}')

        seed_everything(CFG.SEED)

        oof_preds[val_idx] = fold_oof_probs
        oof_labels[val_idx] = fold_val_labels

    # ── OOF 저장 및 T 최적화 ────────────────────────────────────────────────
    oof_path = CFG.CKPT_DIR / 'oof_preds.npy'
    np.save(oof_path, oof_preds)
    np.save(CFG.CKPT_DIR / 'oof_labels.npy', oof_labels)

    oof_ll = log_loss(oof_labels, oof_preds)
    print(f'\n  OOF LogLoss (T=1.0): {oof_ll:.4f}')

    # OOF 기반 최적 T 탐색
    from temperature_scale import apply_temperature
    from scipy.optimize import minimize_scalar
    def obj(T):
        scaled = apply_temperature(oof_preds, T)
        return log_loss(oof_labels, scaled)
    result = minimize_scalar(obj, bounds=(0.15, 2.0), method='bounded')
    T_opt = result.x
    oof_ll_opt = result.fun
    print(f'  OOF 최적 T: {T_opt:.4f} | OOF LogLoss: {oof_ll:.4f} → {oof_ll_opt:.4f}')
    print(f'  OOF predictions 저장: {oof_path}')



# ── Pseudo Labeling ───────────────────────────────────────────────────────────

@torch.no_grad()
def generate_pseudo_labels(video_cache: dict, device: torch.device) -> pd.DataFrame:
    """Phase 1 체크포인트로 test set pseudo labels 생성."""
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
    for col in ['id', 'ID', 'Id']:
        if col in sample_sub.columns:
            test_ids = sample_sub[col].tolist()
            break
    test_df = pd.DataFrame({'id': test_ids})

    fold_preds = []
    for fold in range(1, CFG.N_FOLDS + 1):
        ckpt_path = CFG.CKPT_DIR / f'fold{fold}_phase1.pth'
        if not ckpt_path.exists():
            continue
        model = build_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()

        ds     = StructureDataset(test_df, CFG.TEST_DIR, get_val_transform(CFG.IMAGE_SIZE), is_test=True)
        loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True)
        preds = []
        for views, _ in loader:
            views       = views.to(device)
            video_feats = torch.zeros(views.size(0), CFG.VIDEO_FEAT_DIM, device=device)
            with autocast('cuda', dtype=torch.bfloat16, enabled=CFG.AMP):
                logits = model(views, video_feats=video_feats)
            preds.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        fold_preds.append(np.concatenate(preds, axis=0))
        del model; torch.cuda.empty_cache()

    if not fold_preds:
        return pd.DataFrame(columns=['id', 'label', 'source'])

    ensemble  = np.exp(np.mean([np.log(p + 1e-8) for p in fold_preds], axis=0))
    ensemble  = ensemble / ensemble.sum(axis=1, keepdims=True)
    max_probs = ensemble.max(axis=1)
    pred_lbl  = ensemble.argmax(axis=1)
    high_conf = max_probs >= CFG.PSEUDO_CONF

    idx2label = {0: 'stable', 1: 'unstable'}
    records = [
        {'id': test_ids[i], 'label': idx2label[pred_lbl[i]], 'source': 'pseudo'}
        for i in range(len(test_ids)) if high_conf[i]
    ]
    pseudo_df = pd.DataFrame(records)
    print(f'\n[Pseudo Labeling] {len(pseudo_df)}/{len(test_ids)}개 선택 (conf≥{CFG.PSEUDO_CONF})')
    stable_cnt   = (pseudo_df['label'] == 'stable').sum() if len(pseudo_df) else 0
    unstable_cnt = (pseudo_df['label'] == 'unstable').sum() if len(pseudo_df) else 0
    print(f'  stable={stable_cnt}, unstable={unstable_cnt}')
    return pseudo_df

# ── Phase 2: Domain Fine-Tune ────────────────────────────────────────────────

def train_phase2(video_cache: dict, pseudo_df: pd.DataFrame, device: torch.device, soft_labels: dict = None, only_folds: list = None, no_focal: bool = False):
    combined_df = make_combined_df(CFG.TRAIN_CSV, CFG.DEV_CSV)
    if len(pseudo_df) > 0:
        combined_df = pd.concat([combined_df, pseudo_df], ignore_index=True)

    # ── Phase 2 validation split (stratified 10%) ─────────────────────────────
    from sklearn.model_selection import train_test_split
    p2_train_df, p2_val_df = train_test_split(
        combined_df, test_size=CFG.PHASE2_VAL_RATIO,
        stratify=combined_df['label'], random_state=CFG.SEED
    )

    loss_name = 'CE Loss' if no_focal else 'FocalLoss'
    print(f'\n{"="*60}')
    print(f'  Phase 2 — Domain Fine-Tune  |  Epochs: {CFG.PHASE2_EPOCHS}  |  {loss_name}')
    print(f'  Cosine T_max: {CFG.PHASE2_COSINE_TMAX}  |  patience: {CFG.PHASE2_PATIENCE}')
    print(f'  Train: {len(p2_train_df)}개  Val: {len(p2_val_df)}개')
    if only_folds:
        print(f'  대상 Fold: {only_folds}')
    print(f'{"="*60}')

    for fold in range(CFG.N_FOLDS):
        if only_folds and (fold + 1) not in only_folds:
            print(f'\n── Fold {fold+1}/{CFG.N_FOLDS} ── [스킵]')
            continue
        ckpt_in  = CFG.CKPT_DIR / f'fold{fold+1}_phase1.pth'
        ckpt_out = CFG.CKPT_DIR / f'fold{fold+1}_phase2.pth'
        if not ckpt_in.exists():
            print(f'  Fold {fold+1}: Phase 1 체크포인트 없음, 스킵')
            continue

        print(f'\n── Fold {fold+1}/{CFG.N_FOLDS} Fine-Tune ──')
        model = build_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt_in, map_location=device))
        ema = ModelEMA(model, decay=CFG.EMA_DECAY) if CFG.USE_EMA else None

        # train/val dataset 구성
        def make_p2_ds(df, is_aug):
            trn_part    = df[df['source'] == 'train']
            dev_part    = df[df['source'] == 'dev']
            pseudo_part = df[df['source'] == 'pseudo']
            tfm = get_train_transform(CFG.IMAGE_SIZE) if is_aug else get_val_transform(CFG.IMAGE_SIZE)
            datasets = []
            if len(trn_part) > 0:
                datasets.append(StructureDataset(trn_part, CFG.TRAIN_DIR, tfm,
                                                 video_cache=video_cache, soft_labels=soft_labels))
            if len(dev_part) > 0:
                dev_tfm = get_train_transform(CFG.IMAGE_SIZE) if is_aug else get_val_transform(CFG.IMAGE_SIZE)
                datasets.append(StructureDataset(dev_part, CFG.DEV_DIR, dev_tfm, soft_labels=soft_labels))
            if len(pseudo_part) > 0:
                datasets.append(StructureDataset(pseudo_part, CFG.TEST_DIR, tfm))
            return ConcatDataset(datasets) if datasets else None

        trn_ds = make_p2_ds(p2_train_df, is_aug=True)
        val_ds = make_p2_ds(p2_val_df,   is_aug=False)

        if len(pseudo_df) > 0:
            print(f'  Pseudo {len(pseudo_df)}개 포함')

        trn_loader = DataLoader(trn_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                num_workers=CFG.NUM_WORKERS, pin_memory=True,
                                persistent_workers=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                                num_workers=2, pin_memory=True,
                                persistent_workers=False)

        criterion   = nn.CrossEntropyLoss() if no_focal else FocalLoss(gamma=2.0)
        optimizer   = optim.AdamW(model.parameters(), lr=CFG.PHASE2_LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler   = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.PHASE2_COSINE_TMAX, eta_min=CFG.MIN_LR
        )
        scaler      = GradScaler('cuda', enabled=CFG.AMP)
        resume_path = CFG.CKPT_DIR / f'fold{fold+1}_phase2_resume.pth'
        start_epoch = 1
        best_val    = float('inf')
        patience_cnt = 0

        # ── 중단된 Phase2 재개 ──────────────────────────────────────────────
        if resume_path.exists():
            state = torch.load(resume_path, map_location=device)
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
            start_epoch  = state['epoch'] + 1
            best_val     = state.get('best_val', float('inf'))
            patience_cnt = state.get('patience_cnt', 0)
            print(f'  [Resume] Phase2 Fold {fold+1} epoch {start_epoch}부터 재개 (best_val={best_val:.4f})')

        for epoch in range(start_epoch, CFG.PHASE2_EPOCHS + 1):
            t0       = time.time()
            trn_loss = train_epoch(model, trn_loader, optimizer, criterion, scaler, device, ema=ema)
            eval_model = ema.module if ema is not None else model
            val_loss, _ = val_epoch(eval_model, val_loader, device)
            if epoch <= CFG.PHASE2_COSINE_TMAX:
                scheduler.step()
            # epoch > PHASE2_COSINE_TMAX: LR = MIN_LR 유지

            improved = val_loss < best_val
            mark = ' ★' if improved else ''
            print(f'  Epoch {epoch:02d}/{CFG.PHASE2_EPOCHS} | Train: {trn_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.1f}s{mark}', flush=True)

            if improved:
                best_val = val_loss
                patience_cnt = 0
                save_state = ema.state_dict() if ema is not None else model.state_dict()
                torch.save(save_state, ckpt_out)
            else:
                patience_cnt += 1
                if patience_cnt >= CFG.PHASE2_PATIENCE:
                    print(f'  Early stopping (patience={CFG.PHASE2_PATIENCE})')
                    break

            gc.collect()
            torch.cuda.empty_cache()
            resume_state = {
                'epoch':       epoch,
                'model':       model.state_dict(),
                'optimizer':   optimizer.state_dict(),
                'scheduler':   scheduler.state_dict(),
                'scaler':      scaler.state_dict(),
                'best_val':    best_val,
                'patience_cnt': patience_cnt,
            }
            if ema is not None:
                resume_state['ema'] = ema.state_dict()
            torch.save(resume_state, resume_path)
            del resume_state
            gc.collect()

        if not ckpt_out.exists():
            save_state = ema.state_dict() if ema is not None else model.state_dict()
            torch.save(save_state, ckpt_out)
        print(f'  Best Val: {best_val:.4f} → {ckpt_out}', flush=True)
        resume_path.unlink(missing_ok=True)
        del model, trn_loader, val_loader
        if ema is not None:
            del ema
        gc.collect()
        torch.cuda.empty_cache()


# ── Entry Point ───────────────────────────────────────────────────────────────

def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--folds', type=int, nargs='+', default=None,
                        help='재학습할 fold 번호 (예: --folds 1 4 5). 미지정 시 전체 실행.')
    parser.add_argument('--phase1-only', action='store_true',
                        help='Phase 1만 실행하고 종료.')
    parser.add_argument('--phase2-only', action='store_true',
                        help='Phase 2만 실행 (Phase 1 체크포인트 필요).')
    parser.add_argument('--seed', type=int, default=None,
                        help='랜덤 시드 override (예: --seed 44). 미지정 시 config.py SEED 사용.')
    parser.add_argument('--no-warmup', action='store_true',
                        help='Warmup 없이 CosineAnnealingLR만 사용.')
    parser.add_argument('--no-focal', action='store_true',
                        help='Focal Loss 대신 CE Loss만 사용.')
    args = parser.parse_args()

    if args.seed is not None:
        CFG.SEED = args.seed
    seed_everything(CFG.SEED)
    device = torch.device(CFG.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    # ── Run 폴더 설정 (타임스탬프 기반 버전 관리) ───────────────────────────
    CFG.CKPT_DIR = resolve_run_dir()
    print(f'Checkpoint dir: {CFG.CKPT_DIR}')

    # 영상 특징 캐시 로드 (preprocess_video.py 선행 필요)
    video_cache = load_video_cache(CFG.VIDEO_CACHE)
    if video_cache:
        print(f'영상 특징 로드: {len(video_cache)}개 샘플')
    else:
        print('영상 특징 캐시 없음 — preprocess_video.py를 먼저 실행하세요')

    train_df = pd.read_csv(CFG.TRAIN_CSV)
    dev_df   = pd.read_csv(CFG.DEV_CSV)

    soft_labels = load_soft_labels(CFG.SOFT_LABEL_PATH)
    if soft_labels:
        print(f'Soft Label KD 활성화 (weight={CFG.SOFT_LABEL_WEIGHT})')

    if not args.phase2_only:
        train_phase1(train_df, video_cache, device, soft_labels=soft_labels, only_folds=args.folds,
                     no_warmup=args.no_warmup, no_focal=args.no_focal)
    if not args.phase1_only:
        pseudo_df = generate_pseudo_labels(video_cache, device)
        train_phase2(video_cache, pseudo_df, device, soft_labels=soft_labels,
                     only_folds=args.folds, no_focal=args.no_focal)

    # 완료 마킹 (다음 실행 시 새 run 폴더 생성)
    (CFG.CKPT_BASE_DIR / '.current_run').unlink(missing_ok=True)
    print(f'\n학습 완료. 체크포인트: {CFG.CKPT_DIR}')
    print('python inference.py 실행하여 제출 파일 생성하세요.')


if __name__ == '__main__':
    main()


---
## inference.py — TTA 추론 + fold 앙상블


In [ ]:
"""
추론 파이프라인

1. 모든 fold의 Phase 2 체크포인트 로드 (없으면 Phase 1 fallback)
2. TTA(Test-Time Augmentation): 각 fold × TTA_TIMES 예측 평균
3. 앙상블 결과로 sample_submission 형식에 맞춰 CSV 저장

제출 형식 (확인됨):
    id, unstable_prob, stable_prob
    TEST_0001, 0.15, 0.85
    ...
"""

import argparse
import datetime
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.amp import autocast

from config import CFG
from dataset import StructureDataset, get_val_transform, get_tta_transform
from model import build_model


# ── TTA 추론 ──────────────────────────────────────────────────────────────────

@torch.no_grad()
def predict_tta(model, test_df, device, tta_times: int = CFG.TTA_TIMES) -> np.ndarray:
    """
    TTA_TIMES번 랜덤 aug를 적용한 예측의 평균을 반환한다.
    첫 1회는 aug 없는 clean 예측, 이후는 TTA.
    Returns: (N, NUM_CLASSES) 확률 배열
    """
    model.eval()
    all_rounds = []

    for t in range(tta_times):
        transform = (
            get_val_transform(CFG.IMAGE_SIZE)      # 0번째: clean
            if t == 0
            else get_tta_transform(CFG.IMAGE_SIZE)  # 이후: TTA
        )
        ds = StructureDataset(
            test_df, CFG.TEST_DIR, transform=transform, is_test=True
        )
        loader = DataLoader(
            ds,
            batch_size=CFG.BATCH_SIZE * 2,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True,
        )

        round_probs = []
        for views, _ in loader:
            views       = views.to(device)
            video_feats = torch.zeros(views.size(0), CFG.VIDEO_FEAT_DIM, device=device)
            with autocast('cuda', dtype=torch.bfloat16, enabled=CFG.AMP):
                logits = model(views, video_feats=video_feats)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            round_probs.append(probs)

        all_rounds.append(np.concatenate(round_probs, axis=0))

    # TTA 평균 (log-space averaging이 더 안정적)
    log_probs = [np.log(p + 1e-8) for p in all_rounds]
    avg_log   = np.mean(log_probs, axis=0)
    avg_probs = np.exp(avg_log)
    avg_probs = avg_probs / avg_probs.sum(axis=1, keepdims=True)   # renormalize
    return avg_probs


# ── ID 순서 보존 추론 ─────────────────────────────────────────────────────────

def get_sample_ids(test_df) -> list[str]:
    """test_df에서 ID 컬럼 추출 (컬럼명 유연하게 처리)"""
    for col in ['ID', 'id', 'Id']:
        if col in test_df.columns:
            return test_df[col].tolist()
    raise KeyError('ID 컬럼을 찾을 수 없습니다. sample_submission.csv를 확인하세요.')


# ── 메인 추론 ─────────────────────────────────────────────────────────────────

def resolve_ckpt_dir():
    """
    .current_run 마커 → 해당 run 폴더 사용.
    없으면 checkpoints/ 하위 폴더 중 가장 최신 것 사용.
    """
    base   = CFG.CKPT_BASE_DIR
    marker = base / '.current_run'

    if marker.exists():
        run_id  = marker.read_text().strip()
        run_dir = base / run_id
        if run_dir.exists():
            return run_dir

    # 마커 없으면 타임스탬프 이름(YYYYMMDD_HHMM) 폴더 중 최신 것
    candidates = sorted(
        [d for d in base.iterdir() if d.is_dir() and len(d.name) == 13],
        reverse=True
    )
    if candidates:
        return candidates[0]

    return base  # fallback: 기존 flat 구조



def resolve_ckpt_dirs() -> list:
    """메인 dir + CFG.EXTRA_CKPT_DIRS 합쳐서 반환 (헤테로 앙상블용)."""
    dirs = [resolve_ckpt_dir()]
    from pathlib import Path as _Path
    for extra in CFG.EXTRA_CKPT_DIRS:
        d = _Path(extra)
        if d.exists():
            dirs.append(d)
    return dirs

def run_inference(args=None):
    device  = torch.device(CFG.DEVICE if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    if args is not None and args.ckpt:
        from pathlib import Path as _Path
        ckpt_dirs = [_Path(args.ckpt)]
    else:
        ckpt_dirs = resolve_ckpt_dirs()
    CFG.CKPT_DIR = ckpt_dirs[0]
    print(f'Checkpoint dirs: {ckpt_dirs}')

    # sample_submission으로 test ID 목록 파악
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
    test_ids   = get_sample_ids(sample_sub)
    test_df    = pd.DataFrame({'id': test_ids})

    fold_preds = []

    for ckpt_dir in ckpt_dirs:
      for fold in range(1, CFG.N_FOLDS + 1):
        if args is not None and args.folds and fold not in args.folds:
            print(f'  Fold {fold}: 제외 (--folds)')
            continue

        # Phase 2 체크포인트 우선, 없으면 Phase 1 fallback
        ckpt_p2 = ckpt_dir / f'fold{fold}_phase2.pth'
        ckpt_p1 = ckpt_dir / f'fold{fold}_phase1.pth'

        phase1_only = args is not None and args.phase1_only
        if not phase1_only and ckpt_p2.exists():
            ckpt_path = ckpt_p2
            tag       = 'phase2'
        elif ckpt_p1.exists():
            ckpt_path = ckpt_p1
            tag       = 'phase1'
        else:
            print(f'  Fold {fold}: 체크포인트 없음, 스킵')
            continue

        print(f'\n── Fold {fold} ({tag}) TTA×{CFG.TTA_TIMES} 추론 ──')
        model = build_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        preds = predict_tta(model, test_df, device)
        fold_preds.append(preds)
        print(f'  예측 완료: {preds.shape}')
        del model; torch.cuda.empty_cache()

    if not fold_preds:
        raise RuntimeError('사용 가능한 체크포인트가 없습니다. train.py를 먼저 실행하세요.')

    # Fold 앙상블 (log-space geometric mean)
    log_preds  = [np.log(p + 1e-8) for p in fold_preds]
    ensemble   = np.exp(np.mean(log_preds, axis=0))
    ensemble   = ensemble / ensemble.sum(axis=1, keepdims=True)

    # ── 제출 파일 생성 ────────────────────────────────────────────────────────
    # 실제 제출 컬럼: id, unstable_prob, stable_prob
    # LABEL2IDX: stable=0, unstable=1
    PROB_COL_MAP = {
        'unstable_prob': 1,   # ensemble[:, 1]
        'stable_prob':   0,   # ensemble[:, 0]
        'unstable':      1,
        'stable':        0,
    }

    sub_cols = sample_sub.columns.tolist()
    print(f'\n제출 컬럼: {sub_cols}')

    submission = pd.DataFrame({'id': test_ids})

    for col in sub_cols:
        if col == 'id':
            continue
        if col in PROB_COL_MAP:
            submission[col] = ensemble[:, PROB_COL_MAP[col]]
        else:
            print(f'  ⚠ 알 수 없는 컬럼 "{col}" → 0.5로 채움')
            submission[col] = 0.5

    # 저장
    ts         = datetime.datetime.now().strftime('%m%d_%H%M')
    out_path   = CFG.SUBMIT_DIR / f'submission_{ts}.csv'
    submission.to_csv(out_path, index=False)
    print(f'\n제출 파일 저장 완료: {out_path}')
    print(submission.head(10))
    return submission


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--phase1-only', action='store_true', help='Phase1 체크포인트만 사용')
    parser.add_argument('--folds', type=int, nargs='+', default=None,
                        help='사용할 fold 번호 (예: --folds 1 2 3)')
    parser.add_argument('--ckpt', type=str, default=None,
                        help='체크포인트 dir (예: checkpoints/20260312_0131)')
    args = parser.parse_args()
    run_inference(args)


---
## temperature_scale.py — T-scaling 적용


In [ ]:
"""
Temperature Scaling (dev/OOF 기반 T 최적화 → test submission 적용)

실행:
    python temperature_scale.py                          # dev 기반 최적 T 탐색 후 적용
    python temperature_scale.py --sub submissions/A.csv  # 특정 submission에 적용
    python temperature_scale.py --T 0.465                # T 고정 적용
    python temperature_scale.py --oof                    # OOF 기반 T 탐색 (GPU 불필요)
    python temperature_scale.py --oof --ckpt checkpoints/20260318_2216  # 특정 체크포인트 OOF
"""

import argparse
import datetime
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from torch.amp import autocast
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

from config import CFG
from dataset import StructureDataset, get_val_transform
from model import build_model


@torch.no_grad()
def _predict_single_fold(model, loader, device) -> np.ndarray:
    """단일 fold dev 예측."""
    model.eval()
    preds = []
    for views, _ in loader:
        views = views.to(device)
        video_feats = torch.zeros(views.size(0), CFG.VIDEO_FEAT_DIM, device=device)
        with autocast('cuda', dtype=torch.bfloat16, enabled=CFG.AMP):
            logits = model(views, video_feats)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        preds.append(probs)
    return np.concatenate(preds, axis=0)


@torch.no_grad()
def predict_dev(ckpt_dir: Path, device, folds=None) -> np.ndarray:
    """dev set 예측 (5-fold 앙상블, TTA 없음). folds로 특정 fold만 선택 가능."""
    dev_df = pd.read_csv(CFG.DEV_CSV)
    transform = get_val_transform(CFG.IMAGE_SIZE)
    ds = StructureDataset(dev_df, CFG.DEV_DIR, transform=transform, is_test=True)
    loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

    fold_preds = []
    for fold in range(1, CFG.N_FOLDS + 1):
        if folds and fold not in folds:
            print(f'  Fold {fold}: 제외 (--folds)')
            continue

        ckpt = ckpt_dir / f'fold{fold}_phase2.pth'
        if not ckpt.exists():
            ckpt = ckpt_dir / f'fold{fold}_phase1.pth'
        if not ckpt.exists():
            print(f'  Fold {fold}: 체크포인트 없음, 스킵')
            continue

        model = build_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device))

        preds = _predict_single_fold(model, loader, device)
        fold_preds.append(preds)
        del model
        torch.cuda.empty_cache()
        print(f'  Fold {fold}: 예측 완료')

    if not fold_preds:
        raise RuntimeError('사용 가능한 체크포인트 없음')

    # geometric mean ensemble
    log_preds = [np.log(p + 1e-8) for p in fold_preds]
    ensemble = np.exp(np.mean(log_preds, axis=0))
    ensemble /= ensemble.sum(axis=1, keepdims=True)
    return ensemble


@torch.no_grad()
def predict_dev_per_fold(ckpt_dir: Path, device, folds=None) -> dict:
    """각 fold 개별 dev 예측 반환. {fold_num: preds_array}"""
    dev_df = pd.read_csv(CFG.DEV_CSV)
    transform = get_val_transform(CFG.IMAGE_SIZE)
    ds = StructureDataset(dev_df, CFG.DEV_DIR, transform=transform, is_test=True)
    loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

    results = {}
    for fold in range(1, CFG.N_FOLDS + 1):
        if folds and fold not in folds:
            continue

        ckpt = ckpt_dir / f'fold{fold}_phase2.pth'
        if not ckpt.exists():
            ckpt = ckpt_dir / f'fold{fold}_phase1.pth'
        if not ckpt.exists():
            print(f'  Fold {fold}: 체크포인트 없음, 스킵')
            continue

        model = build_model(pretrained=False).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device))

        preds = _predict_single_fold(model, loader, device)
        results[fold] = preds
        del model
        torch.cuda.empty_cache()

    return results


def apply_temperature(probs: np.ndarray, T: float) -> np.ndarray:
    """log-odds 스케일에서 temperature 적용."""
    log_odds = np.log(probs[:, 1] + 1e-10) - np.log(probs[:, 0] + 1e-10)
    p_unstable = 1.0 / (1.0 + np.exp(-log_odds / T))
    return np.stack([1.0 - p_unstable, p_unstable], axis=1)


def find_optimal_T(probs: np.ndarray, labels: np.ndarray,
                   bounds=(0.05, 3.0)) -> float:
    """LogLoss를 최소화하는 T 탐색."""
    def objective(T):
        scaled = apply_temperature(probs, T)
        return log_loss(labels, scaled)

    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result.x


def bootstrap_T(probs: np.ndarray, labels: np.ndarray,
                n_boot: int = 2000, seed: int = 42) -> dict:
    """Bootstrap으로 최적 T의 신뢰구간 추정.

    Returns:
        dict with keys: T_median, T_mean, T_std, ci_lower, ci_upper,
                        T_samples (full bootstrap distribution)
    """
    rng = np.random.RandomState(seed)
    n = len(labels)
    T_samples = []

    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        try:
            T_i = find_optimal_T(probs[idx], labels[idx])
            T_samples.append(T_i)
        except Exception:
            continue

    T_samples = np.array(T_samples)
    return {
        'T_median': np.median(T_samples),
        'T_mean':   np.mean(T_samples),
        'T_std':    np.std(T_samples),
        'ci_lower': np.percentile(T_samples, 2.5),
        'ci_upper': np.percentile(T_samples, 97.5),
        'T_samples': T_samples,
    }


def run_oof_tscale(ckpt_dir: Path):
    """OOF 기반 T-scaling (GPU 불필요). Bootstrap CI 포함."""
    oof_pred_path = ckpt_dir / 'oof_preds.npy'
    oof_label_path = ckpt_dir / 'oof_labels.npy'

    if not oof_pred_path.exists() or not oof_label_path.exists():
        print(f'OOF 파일 없음: {ckpt_dir}')
        return None

    oof_preds = np.load(oof_pred_path)
    oof_labels = np.load(oof_label_path)

    # 유효한 행만 필터 (zeros = 미완성 fold)
    valid = oof_preds.sum(axis=1) > 0
    n_valid = valid.sum()
    n_total = len(oof_labels)
    print(f'OOF: {n_valid}/{n_total}개 유효')

    if n_valid < 50:
        print(f'  유효 샘플 부족 ({n_valid}개), 신뢰도 낮음')
        if n_valid == 0:
            return None

    probs = oof_preds[valid]
    labels = oof_labels[valid]

    # normalize (합이 1이 아닐 수 있음)
    row_sums = probs.sum(axis=1, keepdims=True)
    probs = probs / np.where(row_sums > 0, row_sums, 1.0)

    # 라벨 분포
    n0, n1 = (labels == 0).sum(), (labels == 1).sum()
    print(f'  라벨 분포: stable={n0}, unstable={n1}')

    # T=1.0 baseline
    ll_base = log_loss(labels, probs)
    print(f'  OOF LogLoss (T=1.0): {ll_base:.6f}')

    # 점추정
    T_opt = find_optimal_T(probs, labels)
    scaled = apply_temperature(probs, T_opt)
    ll_scaled = log_loss(labels, scaled)
    print(f'  점추정 T: {T_opt:.4f} | OOF LogLoss: {ll_base:.6f} → {ll_scaled:.6f}')

    # Bootstrap CI
    print(f'\n  Bootstrap 2000회 진행 중...')
    boot = bootstrap_T(probs, labels)
    print(f'  ──────────────────────────────────────')
    print(f'  T median:  {boot["T_median"]:.4f}')
    print(f'  T mean:    {boot["T_mean"]:.4f} ± {boot["T_std"]:.4f}')
    print(f'  95% CI:    [{boot["ci_lower"]:.4f}, {boot["ci_upper"]:.4f}]')

    # 제출 시 권장 T 범위 (CI 기반, 보수적)
    # 과도한 sharpening 방지: CI 하한과 median 중 큰 값
    T_safe = max(boot['ci_lower'], 0.212)
    T_rec = boot['T_median']
    print(f'\n  ▶ 권장 제출 T: {T_rec:.3f} (범위: {T_safe:.3f} ~ {boot["ci_upper"]:.3f})')

    # Grid search: 주요 T 값별 OOF LogLoss
    print(f'\n  T별 OOF LogLoss:')
    grid_ts = np.arange(0.15, 0.50, 0.01)
    best_grid_t, best_grid_ll = None, float('inf')
    for t in grid_ts:
        sc = apply_temperature(probs, t)
        ll = log_loss(labels, sc)
        if ll < best_grid_ll:
            best_grid_t, best_grid_ll = t, ll
        marker = ' ◀ best' if abs(t - T_opt) < 0.005 else ''
        if 0.20 <= t <= 0.35 or abs(t - T_opt) < 0.005:
            print(f'    T={t:.2f}: {ll:.6f}{marker}')

    return {'T_opt': T_opt, 'boot': boot, 'n_valid': n_valid}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--sub', type=str, default=None, help='적용할 submission CSV')
    parser.add_argument('--T', type=float, default=None, help='T 고정값 (없으면 dev로 탐색)')
    parser.add_argument('--ckpt', type=str, default=None, help='dev 예측용 체크포인트 dir')
    parser.add_argument('--folds', type=int, nargs='+', default=None,
                        help='사용할 fold 번호 (예: --folds 1 2 3)')
    parser.add_argument('--per-fold', action='store_true',
                        help='각 fold 개별 dev LogLoss 출력')
    parser.add_argument('--oof', action='store_true',
                        help='OOF 기반 T 탐색 (GPU 불필요, Bootstrap CI 포함)')
    args = parser.parse_args()

    # ckpt_dir 결정
    ckpt_dir = Path(args.ckpt) if args.ckpt else Path('checkpoints/20260309_1110')

    # ── OOF 모드: GPU 불필요, 저장된 OOF로 T 탐색 ──
    if args.oof:
        print(f'OOF 기반 T-scaling ({ckpt_dir})')
        result = run_oof_tscale(ckpt_dir)
        if result and args.sub:
            T_use = result['boot']['T_median']
            print(f'\n  submission에 T={T_use:.3f} 적용 중...')
            ts = datetime.datetime.now().strftime('%m%d_%H%M')
            sub_path = Path(args.sub)
            sub = pd.read_csv(sub_path)
            probs = sub[['stable_prob', 'unstable_prob']].values
            scaled = apply_temperature(probs, T_use)
            sub['stable_prob'] = scaled[:, 0]
            sub['unstable_prob'] = scaled[:, 1]
            out = CFG.SUBMIT_DIR / f'submission_T{T_use:.3f}_{ts}_{sub_path.stem[-9:]}.csv'
            sub.to_csv(out, index=False)
            print(f'  저장: {out.name}')
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    # ── per-fold 모드: 각 fold 개별 dev LogLoss 출력 후 종료 ──
    if args.per_fold:
        print(f'Per-fold dev 분석 중... ({ckpt_dir})')
        dev_df = pd.read_csv(CFG.DEV_CSV)
        dev_labels = dev_df['label'].map({'stable': 0, 'unstable': 1}).values

        per_fold_preds = predict_dev_per_fold(ckpt_dir, device, folds=args.folds)
        print(f'\n{"="*55}')
        print(f'  Per-Fold Dev LogLoss  |  {ckpt_dir.name}')
        print(f'{"="*55}')
        for fold_num in sorted(per_fold_preds.keys()):
            preds = per_fold_preds[fold_num]
            ll = log_loss(dev_labels, preds)
            T_opt_f = find_optimal_T(preds, dev_labels)
            scaled = apply_temperature(preds, T_opt_f)
            ll_scaled = log_loss(dev_labels, scaled)
            print(f'  Fold {fold_num}: DevLL={ll:.4f} → T={T_opt_f:.3f} DevLL={ll_scaled:.4f}')

        # 앙상블 결과도 출력
        if len(per_fold_preds) > 1:
            all_preds = list(per_fold_preds.values())
            log_preds = [np.log(p + 1e-8) for p in all_preds]
            ensemble = np.exp(np.mean(log_preds, axis=0))
            ensemble /= ensemble.sum(axis=1, keepdims=True)
            ll_ens = log_loss(dev_labels, ensemble)
            T_opt_ens = find_optimal_T(ensemble, dev_labels)
            scaled_ens = apply_temperature(ensemble, T_opt_ens)
            ll_ens_scaled = log_loss(dev_labels, scaled_ens)
            folds_str = ','.join(str(f) for f in sorted(per_fold_preds.keys()))
            print(f'  ────────────────────────────────────────')
            print(f'  Ensemble({folds_str}): DevLL={ll_ens:.4f} → T={T_opt_ens:.3f} DevLL={ll_ens_scaled:.4f}')
        print()
        return

    # 최적 T 탐색
    if args.T is not None:
        T_opt = args.T
        print(f'T 고정: {T_opt}')
    else:
        print(f'Dev 예측 중... ({ckpt_dir})')
        dev_preds = predict_dev(ckpt_dir, device, folds=args.folds)

        dev_df = pd.read_csv(CFG.DEV_CSV)
        dev_labels = dev_df['label'].map({'stable': 0, 'unstable': 1}).values

        before_ll = log_loss(dev_labels, dev_preds)
        print(f'Dev LogLoss (T=1.0): {before_ll:.4f}')

        T_opt = find_optimal_T(dev_preds, dev_labels)
        scaled_dev = apply_temperature(dev_preds, T_opt)
        after_ll = log_loss(dev_labels, scaled_dev)
        print(f'최적 T: {T_opt:.4f} | Dev LogLoss: {before_ll:.4f} → {after_ll:.4f}')

    # submission에 T 적용
    sub_paths = []
    if args.sub:
        sub_paths = [Path(args.sub)]
    else:
        # 최신 숫자 timestamp submission들에 적용 (stacked 제외)
        sub_paths = sorted(
            CFG.SUBMIT_DIR.glob('submission_[0-9]*.csv'),
            key=lambda p: p.stat().st_mtime
        )[-2:]  # 가장 최신 2개 (ConvNeXt + EfficientNetV2-S)

    ts = datetime.datetime.now().strftime('%m%d_%H%M')
    for sub_path in sub_paths:
        sub = pd.read_csv(sub_path)
        probs = sub[['stable_prob', 'unstable_prob']].values
        scaled = apply_temperature(probs, T_opt)
        sub['stable_prob'] = scaled[:, 0]
        sub['unstable_prob'] = scaled[:, 1]
        out = CFG.SUBMIT_DIR / f'submission_T{T_opt:.3f}_{ts}_{sub_path.stem[-9:]}.csv'
        sub.to_csv(out, index=False)
        print(f'저장: {out.name}')

    print('\n완료!')


if __name__ == '__main__':
    main()


---
## utils.py — 유틸리티 함수


In [ ]:
"""
유틸리티: EDA, 데이터 검증, 시각화
"""

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from config import CFG
from dataset import load_views


# ── EDA ──────────────────────────────────────────────────────────────────────

def eda(train_csv: Path, dev_csv: Path):
    train_df = pd.read_csv(train_csv)
    dev_df   = pd.read_csv(dev_csv)

    label_col = 'label' if 'label' in train_df.columns else 'Label'

    print('=' * 50)
    print(f'Train: {len(train_df)}개 | Dev: {len(dev_df)}개')
    print()

    for name, df in [('Train', train_df), ('Dev', dev_df)]:
        counts = df[label_col].value_counts()
        print(f'[{name}] 클래스 분포')
        for cls, cnt in counts.items():
            pct = cnt / len(df) * 100
            print(f'  {cls:10s}: {cnt:4d}개 ({pct:.1f}%)')
        print()

    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (name, df) in zip(axes, [('Train', train_df), ('Dev', dev_df)]):
        counts = df[label_col].value_counts()
        ax.bar(counts.index, counts.values, color=['steelblue', 'tomato'])
        ax.set_title(f'{name} Label Distribution')
        ax.set_ylabel('Count')
        for i, (cls, cnt) in enumerate(counts.items()):
            ax.text(i, cnt + 1, str(cnt), ha='center')
    plt.tight_layout()
    plt.savefig(CFG.ROOT / 'eda_label_dist.png', dpi=120)
    print('EDA 차트 저장: eda_label_dist.png')


def visualize_samples(train_dir: Path, train_csv: Path, n: int = 4):
    """stable / unstable 샘플 각 n개씩 시각화"""
    df        = pd.read_csv(train_csv)
    label_col = 'label' if 'label' in df.columns else 'Label'

    stable_df   = df[df[label_col].str.lower() == 'stable'].sample(n, random_state=42)
    unstable_df = df[df[label_col].str.lower() == 'unstable'].sample(n, random_state=42)
    samples     = pd.concat([stable_df, unstable_df])

    fig, axes = plt.subplots(len(samples), 2, figsize=(8, len(samples) * 3))
    for i, (_, row) in enumerate(samples.iterrows()):
        sid    = row['id']
        label  = row[label_col]
        views  = load_views(train_dir / sid, num_views=2)
        color  = 'green' if label.lower() == 'stable' else 'red'

        for j, img in enumerate(views):
            ax = axes[i][j]
            ax.imshow(img)
            ax.axis('off')
            if j == 0:
                ax.set_title(f'{sid} [{label}]', color=color, fontsize=9)

    plt.suptitle('Sample Visualization (Left=View0, Right=View1)', fontsize=12)
    plt.tight_layout()
    plt.savefig(CFG.ROOT / 'sample_visualization.png', dpi=120)
    print('샘플 시각화 저장: sample_visualization.png')


# ── 데이터 검증 ───────────────────────────────────────────────────────────────

def validate_data(data_dir: Path, csv_path: Path, split: str = 'train'):
    """모든 샘플의 이미지 파일이 존재하는지 확인"""
    df        = pd.read_csv(csv_path)
    label_col = 'label' if 'label' in df.columns else 'Label'
    missing   = []

    for _, row in df.iterrows():
        sample_dir = data_dir / row['id']
        if not sample_dir.exists():
            missing.append(row['ID'])
            continue
        files = []
        for ext in CFG.IMG_EXTENSIONS:
            files.extend(list(sample_dir.glob(f'*{ext}')))
        if len(files) < CFG.NUM_VIEWS:
            missing.append(f"{row['ID']} (이미지 {len(files)}개, 필요 {CFG.NUM_VIEWS}개)")

    print(f'[{split}] 검증 결과: {len(df) - len(missing)}/{len(df)}개 정상')
    if missing:
        print(f'문제 샘플 ({len(missing)}개):')
        for m in missing[:20]:
            print(f'  - {m}')
    else:
        print('모든 샘플 정상.')


# ── 제출 파일 검증 ─────────────────────────────────────────────────────────────

def validate_submission(sub_path: Path, sample_sub_path: Path):
    sub    = pd.read_csv(sub_path)
    sample = pd.read_csv(sample_sub_path)

    print('=== 제출 파일 검증 ===')
    print(f'행 수: {len(sub)} (기대: {len(sample)})')

    # 컬럼 확인
    if list(sub.columns) != list(sample.columns):
        print(f'⚠ 컬럼 불일치: {list(sub.columns)} vs {list(sample.columns)}')
    else:
        print('컬럼 일치: OK')

    # ID 순서 확인
    if list(sub['ID']) != list(sample['ID']):
        print('⚠ ID 순서 불일치')
    else:
        print('ID 순서: OK')

    # 확률값 범위 확인
    prob_cols = [c for c in sub.columns if c != 'ID']
    for col in prob_cols:
        out_of_range = ((sub[col] < 0) | (sub[col] > 1)).sum()
        if out_of_range:
            print(f'⚠ {col}: 범위 벗어난 값 {out_of_range}개')
        else:
            print(f'{col}: 범위 OK  (min={sub[col].min():.4f}, max={sub[col].max():.4f})')

    # 합산 확인
    if len(prob_cols) == 2:
        row_sum = sub[prob_cols].sum(axis=1)
        if not np.allclose(row_sum, 1.0, atol=1e-5):
            print(f'⚠ 행별 합산 != 1 (최대 오차: {(row_sum - 1).abs().max():.6f})')
        else:
            print('행별 합산 = 1.0: OK')

    print('===================')


if __name__ == '__main__':
    eda(CFG.TRAIN_CSV, CFG.DEV_CSV)
    validate_data(CFG.TRAIN_DIR, CFG.TRAIN_CSV, 'train')
    validate_data(CFG.DEV_DIR,   CFG.DEV_CSV,   'dev')
    visualize_samples(CFG.TRAIN_DIR, CFG.TRAIN_CSV)
